# PIV-Q27-P0 v1.7 CROSS-AUDITED DEEP-AUDIT-FIXED — Qwen3.8-27B Stateful Python REPL + Fresh Exact Train Verifier
## ARC-AGI-2 | 4× NVIDIA L4 | Offline Kaggle | Qwen3.8-27B-FP8 TP=4

This notebook is the **first agentic-coding causal experiment after Q1 Direct**. It intentionally changes only the solver mechanism while preserving the proven Q1 infrastructure: same model/checkpoint, vLLM 0.19.0 TP=4 engine, frozen Proxy-48, EDA features, dependency parity, runtime reserve, checkpoint fingerprinting, provenance and scoring denominator.

### Source basis

Primary source snapshot: `gutfeeling/arc-agi-2-submission` at commit
`b3d4494892c6d85fa2a35d548271fb02ea3e191b`.

The P0 mechanism is source-faithful at the **semantic/invariant level**:

1. one reasoning-model lineage per ARC task;
2. one persistent Python REPL namespace across tool turns;
3. the puzzle is preloaded into the REPL and protected from overwrite/whole-puzzle printing;
4. Python stdout/errors/timeouts become model feedback;
5. the model must finish with one self-contained `solution(grid)` program;
6. the final program is executed again in a **fresh process** and must match **every training pair exactly** before any test output is admitted;
7. test grids are produced by execution of the verified program, never transcribed from prose.

### Explicit LOCAL adaptations for Kaggle 4×L4

- Direct `vllm.LLM.generate(...)` + Qwen3.8 native tool-capable chat template instead of an OpenAI/Responses server.
- Lightweight persistent local Python subprocess instead of Docker/IPyBox.
- Hard process timeout + restricted dangerous filesystem/process APIs.
- Bounded P0 budget: one lineage, 6 Python calls, at most 7 model turns, 2,048 tokens/turn and 12,288 generated tokens/task.
- Tool output sent back to Qwen is compact; full bounded raw tool events are checkpointed.
- No reviewer, no second lineage, no deterministic PTQ repair loop, no NVARC seeding, no judge and no evolution. Those are later experiments.

Default is `PREFLIGHT`. Run `PREFLIGHT -> SMOKE_GPU -> PROXY48` in fresh Kaggle sessions.


### v1.1 source/donor re-audit hardening

After re-comparing P0 against the frozen Q1 donor and the audited PiVOTools/Qwen sources, v1.1 fixes four issues without adding a new solver mechanism:
1. use the full source agentic prompt family rather than a shortened hybrid prompt;
2. reject length-truncated generations before parsing them as a tool/final answer;
3. use per-request generation caps inside a batch so one near-context-limit task cannot starve the whole wave;
4. harden the lightweight local worker against host/Kaggle filesystem and process/network access.

It also validates the actual mounted Qwen multi-turn tool-history rendering during PREFLIGHT.


### v1.2 additional verifier/accounting correction

A second source-level pass found that PiVOTools verifies training examples first and executes test inputs in a separate step. v1.1's single verifier exception boundary could erase a genuinely train-perfect result if test execution later failed. v1.2 separates those states and counts train-perfect rate over the entire frozen active task set, including missing/failed tasks as non-passes.


### v1.3 external-review verification

An independent review of the earlier v1.0 was checked against this v1.2 notebook and the audited PiVOTools source.
Some findings had already been fixed in v1.2 (per-request SamplingParams, multi-tool handling, finish-reason gate,
full source prompt, Qwen history probe, train-vs-test verifier state). The remaining real issues are fixed here:
kernel-level `copy` import, source-style bounded retries, a less starvation-prone per-turn ceiling, source-tolerant grid
normalization, per-test partial output retention, budget-aware finalization, think-aware action parsing, dead-worker restart,
puzzle reset hardening, narrower dunder blocking, `py` fences, and a scripted CPU preflight agent-loop test.

Two review suggestions are intentionally NOT adopted:
- no raw-text continuation after truncation (the PiVOTools source retries the agentic call; it does not splice partial text);
- no switch from XHIGH to MEDIUM (PIV-Q27-P0 is the planned XHIGH causal experiment).


### v1.4 second external-review re-audit

This revision corrects the remaining mismatch between the local vLLM loop and PiVOTools' two-level retry semantics. Incomplete/length-capped steps are retried first with the same conversation and persistent REPL state; only after step retries are exhausted may the outer source-style attempt restart.

The local per-turn ceiling now equals the remaining total P0 generation budget (subject to context headroom), the closest bounded analogue to the source configuration's lack of an explicit per-step output-token cap.

P0 still uses one Qwen XHIGH lineage and the same 12,288 total generated-token budget. `SMOKE_GPU` now uses the same generation/tool budget as `PROXY48`, so it is a real viability gate: if first-turn length truncation exceeds 50%, do not launch Proxy-48 without revisiting budget/effort/context policy in a separate revision.


### v1.5 third external-review re-audit

The v1.4 first-turn-length SMOKE gate was too weak. v1.5 gates on terminal budget death and empirical wall-clock projection from real generation waves.


### v1.6 deep-audit fixes (mechanism unchanged)

Third-review fixes, all LOCAL harness hygiene — the solver mechanism (one Qwen XHIGH lineage, persistent REPL,
6 executed Python calls, 12 turns, 12,288 total generated tokens, fresh all-train exact verifier, frozen Proxy-48,
Q1-frozen engine) is unchanged:

1. **Safety reserve 30 → 60 min + per-wave projected-time pre-check.** A single wave can run for the remaining
   budget (up to 12,288 tokens ≈ 33 min at the Q1-measured step latency), so a wave starting just before the old
   30-min deadline could cross the 12 h Kaggle limit and lose `/kaggle/working`. The loop now projects each wave
   from measured wave timings (`wave_metrics.jsonl`) and refuses to start one that would cross the deadline
   (`global_deadline_projected`, resumable).
2. **Stratified SMOKE8.** The previous selector took all four Proxy-48 tier-3 tasks (50% vs 8% in Proxy-48), which
   biased the feasibility gate. SMOKE8 now mirrors Proxy-48: 1 tier-3, 2 tier-2, 5 tier-1 (incl. easy_static,
   hard_tier1, multi_query, dynamic), deterministic in Proxy-48 order.
3. **Structured malformed-tool retry counter** resets only after a VALID structured tool turn is accepted (the
   early reset let the max-2 inner retry be bypassed).
4. **Guard-rejected snippets are not charged** to the 6-call tool budget (they never reached Python); fenced
   ```` ```python ```` parameter payloads are unwrapped before the guard.
5. **Tool output is sanitized** before it re-enters the prompt (`<|im_end|>`, `<tool_response>` … cannot break
   the chat structure).
6. **Length-capped final turns with a CLOSED, parseable `solution()` block are accepted** (LOCAL; the block is
   demonstrably complete, retrying would only burn budget).
7. `malformed_tool` requires an actual `<tool_call>`; prose merely mentioning `<function=` is no longer a retry.
8. SMOKE gate additionally requires complete terminal coverage and reports measured step latency + projected
   max wave minutes; `global_deadline_projected` counts as an infrastructure failure.
9. Stale `per-turn 4096` provenance string corrected.
10. PREFLIGHT now drives the **real** `run_real_piv()` scheduler with a scripted fake LLM (happy path, budget
    death, malformed-tool retry accounting, guard-rejection accounting, length salvage, deadline pre-check).

Optional throughput switch (default OFF to keep the Q1 engine fingerprint): `ENABLE_PREFIX_CACHING`.

### v1.7 cross-audit hardening

Independent re-audit confirmed the v1.6 deep-audit changes, but found one scheduler accounting bug:
inside a multi-call assistant turn, an early guard-rejected call did not increment `tool_calls`, yet later calls
were denied by their original positional index. Thus a rejected call could indirectly consume a valid remaining
execution slot. v1.7 makes the budget test dynamic per call.

Two defensive changes do not alter the PIV mechanism:
- SMOKE terminal coverage also requires unique task IDs;
- projected-deadline timing never becomes less conservative than the Q1-derived 0.20 s/output-token prior.


In [1]:
# =====================================================================================
# CELL 1 — PIV-Q27-P0 CONFIG + OFFLINE ENVIRONMENT + FROZEN SOURCE CONTRACT
# Must run before importing torch/vLLM.
# =====================================================================================
import os
import sys
import json
import math
import time
import random
import hashlib
import zipfile
import shutil
import traceback
import copy
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from collections import deque

SEED = 42
random.seed(SEED)
EXPERIMENT_ID = "PIV-Q27-P0-v1.7"
GENERATION_CONTRACT_VERSION = "piv-q27-p0-agentic-v1.7"
PARSER_CONTRACT_VERSION = "piv-q27-toolxml-finalcode-v1.7"
PROMPT_CONTRACT_VERSION = "piv-q27-source-agentic-prompt-v1.7"

# -------------------------------------------------------------------------------------
# FROZEN SOURCE AUDIT
# -------------------------------------------------------------------------------------
SOURCE_FILE_SHA256 = {
    "qwen_exp1_base": "07d7c7a7ca4567a6118ae8cef00b10277aa37d5ada8698afb66afeb43c88aeb1",
    "qwen_runtime_benchmark": "7d91965196ec3309bf4e3f16f90efcaf2067ee8575d245c8eab9eb100f204b78",
    "arc2_advanced_eda": "1a9384d4b2e01fc15cddad95dee1d6eb5a633b6ae2d041d3d9ae97ffd1879f64",
    "nvarc_31_experiment_diagnostics": "69a64354b6b1c696af1f1c7a68984275d4db9b7eef5f2a382f879dc862d7c978",
    "portability_notes_v10": "c2dd4b45e3697d298d8f814b4087cee742ef4f8632f0eac318ba16cb8f8abda8",
    "experiment_master_plan": "8ca1c34c2c3d864c4b8e04fadf64465cbf468a09b3647c690f7306a07daf0c42"
}

PROXY48_TASK_IDS = [
    "0934a4d8",
    "20a9e565",
    "4c7dc4dd",
    "89565ca0",
    "136b0064",
    "291dc1e1",
    "16b78196",
    "271d71e2",
    "28a6681f",
    "2b83f449",
    "2c181942",
    "31f7f899",
    "35ab12c3",
    "3dc255db",
    "409aa875",
    "4c416de3",
    "64efde09",
    "6e453dd6",
    "7491f3cf",
    "7b0280bc",
    "7b80bb43",
    "7c66cb00",
    "1818057f",
    "332f06d7",
    "53fb4810",
    "7666fa5d",
    "da515329",
    "8f3a5a89",
    "db695cfb",
    "97d7923e",
    "de809cff",
    "c7f57c3e",
    "cb2d8a2c",
    "58490d8a",
    "dbff022c",
    "5961cc34",
    "80a900e0",
    "269e22fb",
    "8b9c3697",
    "e376de54",
    "b99e7126",
    "c4d067a0",
    "135a2760",
    "4a21e3da",
    "71e489b6",
    "800d221b",
    "dd6b8c4b",
    "36a08778"
]
PROXY48_ORDERED_SHA256 = hashlib.sha256("\n".join(PROXY48_TASK_IDS).encode("utf-8")).hexdigest()
PROXY48_EXPECTED_SHA256 = "8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134"
PROXY48_SOURCE_COUNTS = {
    "total": 48,
    "tier1": 32,
    "tier2": 12,
    "tier3": 4,
    "easy_static": 16,
    "rare": 30,
    "multi_query": 8,
    "dynamic": 8,
    "hard_tier1": 2,
    "high_ratio": 5,
    "extreme_ratio": 4,
}

if len(PROXY48_TASK_IDS) != 48 or len(set(PROXY48_TASK_IDS)) != 48:
    raise RuntimeError("Frozen Proxy-48 must contain exactly 48 unique task IDs.")
if PROXY48_ORDERED_SHA256 != PROXY48_EXPECTED_SHA256:
    raise RuntimeError("Frozen Proxy-48 fingerprint mismatch. Refusing to run a changed suite.")

# NVARC notebook diagnostic anchors. These are SOURCE diagnostics only, never Q1 results.
NVARC_DIAGNOSTIC_ANCHORS = {
    "outputs_analyzed": 172,
    "gt_present_in_pool": 55,
    "gt_pool_coverage": 55/172,
    "gt_selected_by_current_dual": 45,
    "current_dual_score_reported": 33.50,
    "oracle_score_reported": 41.17,
    "avg_raw_duplicate_ratio": 0.378,
    "scorer_rank_correlation_on_gt_present": 0.961,
    "purpose": "Motivate complementary candidate coverage and provenance-aware dedup; not used to tune Q1 predictions.",
}


# =====================================================================================
# ONE-LINE RUN SWITCH
# =====================================================================================
RUN_PROFILE = "PREFLIGHT"   # "PREFLIGHT" | "SMOKE_GPU" | "PROXY48"
RESET_PROXY48_OUTPUTS = False

PIV_SOURCE_COMMIT = "b3d4494892c6d85fa2a35d548271fb02ea3e191b"
PIV_SOURCE_REPO = "https://github.com/gutfeeling/arc-agi-2-submission"
PIV_EXTERNAL_SOURCES = {
    "technical_article": "https://pivotools.github.io/pivotools-quarto-blog/posts/agentic_coding_arc_agi/",
    "audited_repo_snapshot": f"https://github.com/gutfeeling/arc-agi-2-submission/tree/{PIV_SOURCE_COMMIT}",
    "public_experiment_traces": "https://huggingface.co/datasets/arcagi2/arcagi2-agentic-coding-publication",
    "qwen_model_card": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8",
    "qwen_chat_template": "https://huggingface.co/Qwen/Qwen3.8-27B-FP8/blob/main/chat_template.jinja",
}
PIV_SOURCE_FILES = {
    "clients.py": {"sha": "34a85ed34001923a5efae715f509e5aa7e8e235f", "role": "persistent model/tool loop + reasoning/tool history"},
    "solver.py": {"sha": "cf2ef554cc389322a5689988b300dc2bc05bf36a", "role": "one code turn -> fresh all-train verification -> test execution"},
    "turn.py": {"sha": "fe4dfc69c5963a56dfba84d0bd49821ae0d2e0bc", "role": "last code block + required solution() artifact"},
    "tools/repl_tool.py": {"sha": "691839335b1e65ba0bd2e4b8b84b70c50d559683", "role": "python tool, errors/timeouts as feedback, puzzle protection"},
    "utils/puzzle_utils.py": {"sha": "7e6ec3034e0dc2f94fba3aa9f1d1d4a0896120b4", "role": "all-train/all-test-input serialization; test outputs hidden"},
    "utils/solving_utils.py": {"sha": "197be842b3d6fa08e4e98066e801dc8cf2bc8327", "role": "fresh sandbox train check and test execution"},
    "utils/code_templates/check_solution_on_training_examples.py": {"sha": "c13ea5b89360ea997a30a1eabe2ccb4a67489b44", "role": "deep-copy input + exact train equality"},
    "solver/prompts/agentic_coding_solver.txt": {"sha": "dab87e08bbe3f071920f13eb98f4a4d6f03ada10", "role": "ARC priors, preloaded puzzle, self-contained solution()"},
    "solver/prompts/system_prompt.txt": {"sha": "265131a50fb42deebe0a8da353f466b11d6d7d6b", "role": "stateful Python tool contract"},
    "solver/config/gpt_oss_120b_high.py": {"sha": "8fd563ddcb10962c7564fbfa8cbf3424a7a437d7", "role": "source agentic config example"},
    "sandbox/ipybox_sandbox.py": {"sha": "195e0e66ad7328a4f823974f04778b639a16412a", "role": "source Docker/IPyBox persistent REPL implementation; semantics only, not ported literally"},
    "sandbox/base.py": {"sha": "517f5794ff0cac64024e47c184d116b1f36577ea", "role": "source stateful REPL/Sandbox interface and fresh run_cells semantics"},
    "evaluation/evaluate.py": {"sha": "8b48fb4482dc7d9772ac010ba6dd81f6cc15e0a4", "role": "source multi-sample evaluation / majority aggregation; not copied into P0"},
    "utils/code_templates/produce_output_grid.py": {"sha": "b6ab435ca4f7d77414764e3a4400bb8c6dbead5e", "role": "execute verified solution on all test inputs"},
}

PROFILE_TABLE = {
    "PREFLIGHT": {
        "use_model": False, "task_set": "PROXY48", "active_tasks": 48,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_P0_PREFLIGHT_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": False,
    },
    "SMOKE_GPU": {
        "use_model": True, "task_set": "SMOKE8", "active_tasks": 8,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_P0_SMOKE_GPU_BUNDLE.zip",
        "reset_outputs": True, "accuracy_is_real": False,
    },
    "PROXY48": {
        "use_model": True, "task_set": "PROXY48", "active_tasks": 48,
        "agent_batch_size": 8, "max_tool_calls": 6, "max_model_turns": 12,
        "per_turn_max_tokens": 12288, "total_generated_token_cap": 12288,
        "max_retries": 2,
        "tool_timeout_sec": 10.0, "verifier_timeout_sec": 20.0,
        "bundle_name": "PIV_Q27_P0_PROXY48_BUNDLE.zip",
        "reset_outputs": RESET_PROXY48_OUTPUTS, "accuracy_is_real": True,
    },
}
RUN_PROFILE = str(RUN_PROFILE).strip().upper()
if RUN_PROFILE not in PROFILE_TABLE:
    raise ValueError(f"RUN_PROFILE={RUN_PROFILE!r} invalid; choose {list(PROFILE_TABLE)}")
PROFILE = PROFILE_TABLE[RUN_PROFILE]
USE_MODEL = bool(PROFILE["use_model"])
TASK_SET = str(PROFILE["task_set"])
ACTIVE_TASK_LIMIT = int(PROFILE["active_tasks"])
AGENT_BATCH_SIZE = int(PROFILE["agent_batch_size"])
MAX_TOOL_CALLS = int(PROFILE["max_tool_calls"])
MAX_MODEL_TURNS = int(PROFILE["max_model_turns"])
MAX_RETRIES = int(PROFILE.get("max_retries", 2))
PER_TURN_MAX_TOKENS = int(PROFILE["per_turn_max_tokens"])
TOTAL_GENERATED_TOKEN_CAP = int(PROFILE["total_generated_token_cap"])
TOOL_TIMEOUT_SEC = float(PROFILE["tool_timeout_sec"])
VERIFIER_TIMEOUT_SEC = float(PROFILE["verifier_timeout_sec"])
BUNDLE_NAME = str(PROFILE["bundle_name"])
ACCURACY_IS_REAL = bool(PROFILE["accuracy_is_real"])

MODEL_PATH = Path("/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1")
WHEELHOUSE = Path("/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3")
ARC_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
TRAIN_CHALLENGES_PATH = ARC_ROOT / "arc-agi_training_challenges.json"
TRAIN_SOLUTIONS_PATH  = ARC_ROOT / "arc-agi_training_solutions.json"
EVAL_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS_PATH   = ARC_ROOT / "arc-agi_evaluation_solutions.json"
TEST_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_test_challenges.json"
SAMPLE_SUBMISSION_PATH = ARC_ROOT / "sample_submission.json"

BASE_WORK_ROOT = Path("/kaggle/working/piv_q27_p0_v17")
WORK_ROOT = BASE_WORK_ROOT / RUN_PROFILE.lower()
if PROFILE["reset_outputs"] and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NOTEBOOK_WALL_START = time.perf_counter()
TOTAL_RUNTIME_SEC = 12 * 60 * 60
# v1.6: one wave may run for the remaining P0 budget (up to 12,288 tokens ~= 33 min at the Q1-measured
# ~0.16 s/step). The reserve must exceed the longest possible wave plus bundle time, and each wave is
# additionally pre-checked against a projection built from measured wave timings.
SAFETY_RESERVE_SEC = 60 * 60
DEFAULT_SEC_PER_OUTPUT_TOKEN = 0.20   # projection prior before the first measured wave
WAVE_TIME_SAFETY_FACTOR = 1.25
WAVE_OVERHEAD_SEC = 90.0              # prefill / scheduling / tool + verifier allowance per wave

# Frozen from the successful Q1 4×L4 runtime.
MAX_MODEL_LEN = 29696
MAX_NUM_BATCHED_TOKENS = 32768
MAX_NUM_SEQS = 16
GPU_MEMORY_UTILIZATION = 0.90
# v1.6 LOCAL throughput option (algorithm-neutral): prefix caching lets every multi-turn prompt reuse the
# already-computed conversation prefix instead of re-prefilling it each turn. Default False keeps the
# Q1-frozen engine fingerprint. Set True after SMOKE if the projected wall-clock is tight; if vLLM refuses
# it for this hybrid-attention checkpoint, revert to False.
ENABLE_PREFIX_CACHING = False

# EXP-0 diagnostics preserved unchanged.
WEIGHT_CLIP_LOW = 0.20
WEIGHT_CLIP_HIGH = 5.00
N_PROPENSITY_FOLDS = 5
HIGH_RATIO_THRESHOLD = 10.0
LARGE_TEST_INPUT_CELLS = 300
MANY_COLORS_THRESHOLD = 7
LOW_BG_THRESHOLD = 0.30

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("=" * 116)
print(EXPERIMENT_ID, "| Stateful Python REPL + Fresh Exact Train Verifier")
print("=" * 116)
print("RUN_ID:", RUN_ID, "| RUN_PROFILE:", RUN_PROFILE, "| TASK_SET:", TASK_SET)
print("PIV source commit:", PIV_SOURCE_COMMIT)
print("Agent budget: calls/turns/per-turn/total =", MAX_TOOL_CALLS, MAX_MODEL_TURNS, PER_TURN_MAX_TOKENS, TOTAL_GENERATED_TOKEN_CAP)
print("Proxy-48 SHA256:", PROXY48_ORDERED_SHA256)
print("WORK_ROOT:", WORK_ROOT)

required = [MODEL_PATH, WHEELHOUSE, ARC_ROOT, TRAIN_CHALLENGES_PATH, TRAIN_SOLUTIONS_PATH,
            EVAL_CHALLENGES_PATH, EVAL_SOLUTIONS_PATH, TEST_CHALLENGES_PATH, SAMPLE_SUBMISSION_PATH]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Required Kaggle input is missing: {p}")
config_path = MODEL_PATH / "config.json"
index_path = MODEL_PATH / "model.safetensors.index.json"
with open(config_path, "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
qcfg = model_cfg.get("quantization_config", {})
if model_cfg.get("model_type") != "qwen3_5":
    raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
    raise RuntimeError(f"Unexpected quantization_config={qcfg}")

def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

def stable_json_sha256(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def small_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_RUNTIME_PACKAGES = {"vllm":"0.19.0", "transformers":"4.57.6", "tokenizers":"0.22.2", "huggingface-hub":"0.36.2"}
_loaded_module_names = {"vllm":"vllm", "transformers":"transformers", "tokenizers":"tokenizers", "huggingface-hub":"huggingface_hub"}
before_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for _pkg, _module in _loaded_module_names.items():
    if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _module in sys.modules:
        raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart Kaggle session before Run All.")

def offline_pip_install(requirements):
    if not requirements: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])
if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
    offline_pip_install(["vllm==0.19.0"])
else:
    repair=[]
    for package in ["transformers","tokenizers","huggingface-hub"]:
        if before_stack[package] != EXPECTED_RUNTIME_PACKAGES[package]:
            module_name = package.replace("-", "_")
            if module_name in sys.modules or (package=="huggingface-hub" and "huggingface_hub" in sys.modules):
                raise RuntimeError(f"{package} already imported with wrong version; restart session.")
            repair.append(f"{package}=={EXPECTED_RUNTIME_PACKAGES[package]}")
    offline_pip_install(repair)
after_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
for package, wanted in EXPECTED_RUNTIME_PACKAGES.items():
    if after_stack.get(package) != wanted:
        raise RuntimeError(f"Runtime parity failure: {package}={after_stack.get(package)!r}, expected {wanted!r}")
print("Runtime dependency parity PASS:", after_stack)


PIV-Q27-P0-v1.7 | Stateful Python REPL + Fresh Exact Train Verifier
RUN_ID: 20260914T121214Z | RUN_PROFILE: PREFLIGHT | TASK_SET: PROXY48
PIV source commit: b3d4494892c6d85fa2a35d548271fb02ea3e191b
Agent budget: calls/turns/per-turn/total = 6 12 12288 12288
Proxy-48 SHA256: 8f432c60a35091dcda2b890574b1f4d3c03aaf48a5b3b7f96b4e88c35d359134
WORK_ROOT: /kaggle/working/piv_q27_p0_v17/preflight
Looking in links: /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/vllm-0.19.0-cp38-abi3-manylinux_2_31_x86_64.whl
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/transformers-4.57.6-py3-none-any.whl (from vllm==0.19.0)
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (from vllm==0.19.0)
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.wh

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.


Runtime dependency parity PASS: {'vllm': '0.19.0', 'transformers': '4.57.6', 'tokenizers': '0.22.2', 'huggingface-hub': '0.36.2'}


In [2]:

# =====================================================================================
# CELL 2 — LOAD ARC DATA
# =====================================================================================
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(SEED)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_challenges = load_json(TRAIN_CHALLENGES_PATH)
train_solutions  = load_json(TRAIN_SOLUTIONS_PATH)
eval_challenges  = load_json(EVAL_CHALLENGES_PATH)
eval_solutions   = load_json(EVAL_SOLUTIONS_PATH)
test_challenges  = load_json(TEST_CHALLENGES_PATH)
sample_submission = load_json(SAMPLE_SUBMISSION_PATH)

assert set(eval_challenges) == set(eval_solutions)
assert set(test_challenges) == set(sample_submission)
missing_proxy_ids = [k for k in PROXY48_TASK_IDS if k not in eval_challenges or k not in eval_solutions]
if missing_proxy_ids:
    raise RuntimeError(f"Frozen Proxy-48 IDs missing from public Evaluation data: {missing_proxy_ids}")

print("Training tasks       :", len(train_challenges))
print("Evaluation tasks     :", len(eval_challenges))
print("Evaluation outputs   :", sum(len(v) for v in eval_solutions.values()))
print("Test tasks           :", len(test_challenges))
print("Visible test queries :", sum(len(v["test"]) for v in test_challenges.values()))


Training tasks       : 1000
Evaluation tasks     : 120
Evaluation outputs   : 172
Test tasks           : 240
Visible test queries : 259


In [3]:

# =====================================================================================
# CELL 3 — EXP-0 STRUCTURAL FEATURES + TEST-RELEVANCE WEIGHTS
# Recomputed here so EXP-1 is self-contained.
# =====================================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, log_loss

def arr(grid):
    return np.asarray(grid, dtype=np.int16)

def is_rectangular_grid(grid, max_hw=30):
    if not isinstance(grid, list) or len(grid) < 1 or len(grid) > max_hw:
        return False
    if not all(isinstance(row, list) for row in grid):
        return False
    widths = [len(row) for row in grid]
    if min(widths, default=0) < 1 or max(widths, default=0) > max_hw or len(set(widths)) != 1:
        return False
    for row in grid:
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, np.integer)):
                return False
            if int(x) < 0 or int(x) > 9:
                return False
    return True

def grid_basic_stats(grid):
    a = arr(grid)
    vals, counts = np.unique(a, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())
    return {
        "h": int(a.shape[0]),
        "w": int(a.shape[1]),
        "cells": int(a.size),
        "n_colors": int(len(vals)),
        "bg0_density": float(np.mean(a == 0)),
        "color_entropy": entropy,
    }

def colored_components_count(grid):
    # Diagnostic only: 4-connected same-color non-zero components.
    a = arr(grid)
    h, w = a.shape
    seen = np.zeros((h, w), dtype=bool)
    n = 0
    sizes = []
    for r in range(h):
        for c in range(w):
            if seen[r, c] or a[r, c] == 0:
                continue
            color = a[r, c]
            q = deque([(r, c)])
            seen[r, c] = True
            size = 0
            while q:
                rr, cc = q.popleft()
                size += 1
                for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                    nr, nc = rr+dr, cc+dc
                    if 0 <= nr < h and 0 <= nc < w and not seen[nr, nc] and a[nr, nc] == color:
                        seen[nr, nc] = True
                        q.append((nr, nc))
            n += 1
            sizes.append(size)
    return n, (float(np.mean(sizes)) if sizes else 0.0)

def summarize(values, prefix):
    x = np.asarray(values, dtype=float)
    if len(x) == 0:
        return {
            f"{prefix}_mean": 0.0, f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0, f"{prefix}_max": 0.0,
        }
    return {
        f"{prefix}_mean": float(x.mean()),
        f"{prefix}_std": float(x.std()),
        f"{prefix}_min": float(x.min()),
        f"{prefix}_max": float(x.max()),
    }

def task_features(task_id, task, split):
    train = task["train"]
    tests = task["test"]

    tr_in_stats, tr_out_stats, te_in_stats = [], [], []
    tr_in_components, tr_out_components, te_components = [], [], []
    tr_in_component_sizes, tr_out_component_sizes, te_component_sizes = [], [], []
    ratios, h_ratios, w_ratios, same_shape = [], [], [], []
    all_visible_colors = set()

    for ex in train:
        si = grid_basic_stats(ex["input"])
        so = grid_basic_stats(ex["output"])
        tr_in_stats.append(si); tr_out_stats.append(so)
        ci, csi = colored_components_count(ex["input"])
        co, cso = colored_components_count(ex["output"])
        tr_in_components.append(ci); tr_out_components.append(co)
        tr_in_component_sizes.append(csi); tr_out_component_sizes.append(cso)
        ratios.append(so["cells"] / max(si["cells"], 1))
        h_ratios.append(so["h"] / max(si["h"], 1))
        w_ratios.append(so["w"] / max(si["w"], 1))
        same_shape.append(int((si["h"], si["w"]) == (so["h"], so["w"])))
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())
        all_visible_colors.update(np.unique(arr(ex["output"])).tolist())

    for ex in tests:
        st = grid_basic_stats(ex["input"])
        te_in_stats.append(st)
        ct, cst = colored_components_count(ex["input"])
        te_components.append(ct); te_component_sizes.append(cst)
        all_visible_colors.update(np.unique(arr(ex["input"])).tolist())

    feat = {
        "task_id": task_id,
        "split": split,
        "n_train": len(train),
        "n_test_queries": len(tests),
        "visible_n_colors": len(all_visible_colors),
        "same_shape_frac": float(np.mean(same_shape)),
        "dynamic_any": int(any(x == 0 for x in same_shape)),
        "multi_query": int(len(tests) > 1),
    }

    for key in ["h", "w", "cells", "n_colors", "bg0_density", "color_entropy"]:
        feat.update(summarize([x[key] for x in tr_in_stats], f"tr_in_{key}"))
        feat.update(summarize([x[key] for x in tr_out_stats], f"tr_out_{key}"))
        feat.update(summarize([x[key] for x in te_in_stats], f"te_in_{key}"))

    feat.update(summarize(ratios, "area_ratio"))
    feat.update(summarize(h_ratios, "h_ratio"))
    feat.update(summarize(w_ratios, "w_ratio"))
    feat.update(summarize(tr_in_components, "tr_in_components"))
    feat.update(summarize(tr_out_components, "tr_out_components"))
    feat.update(summarize(te_components, "te_in_components"))
    feat.update(summarize(tr_in_component_sizes, "tr_in_component_size"))
    feat.update(summarize(tr_out_component_sizes, "tr_out_component_size"))
    feat.update(summarize(te_component_sizes, "te_in_component_size"))

    feat["n_train_eq_2"] = int(len(train) == 2)
    feat["n_train_ge_3"] = int(len(train) >= 3)
    feat["high_ratio"] = int(max(ratios) >= HIGH_RATIO_THRESHOLD)
    feat["large_test_input"] = int(max(x["cells"] for x in te_in_stats) >= LARGE_TEST_INPUT_CELLS)
    feat["many_colors"] = int(len(all_visible_colors) >= MANY_COLORS_THRESHOLD)
    feat["low_bg_train_input"] = int(
        np.mean([x["bg0_density"] for x in tr_in_stats]) < LOW_BG_THRESHOLD
    )
    return feat

def build_feature_df(data, split):
    return pd.DataFrame(
        [task_features(k, v, split) for k, v in data.items()]
    ).sort_values("task_id").reset_index(drop=True)

train_feat = build_feature_df(train_challenges, "train")
eval_feat  = build_feature_df(eval_challenges, "eval")
test_feat  = build_feature_df(test_challenges, "test")

for name, df in [("train", train_feat), ("eval", eval_feat), ("test", test_feat)]:
    df.to_csv(WORK_ROOT / f"task_features_{name}.csv", index=False)

numeric_cols = [
    c for c in eval_feat.columns
    if c not in {"task_id", "split"} and pd.api.types.is_numeric_dtype(eval_feat[c])
]
EXCLUDE_PROPENSITY = {
    "task_id", "split",
    "n_train_eq_2", "n_train_ge_3", "high_ratio", "large_test_input",
    "many_colors", "low_bg_train_input", "multi_query",
}
prop_cols = [c for c in numeric_cols if c not in EXCLUDE_PROPENSITY]

combined = pd.concat(
    [eval_feat.assign(_is_test=0), test_feat.assign(_is_test=1)],
    ignore_index=True
)
X = combined[prop_cols].astype(float)
y = combined["_is_test"].astype(int).to_numpy()

prop_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        C=0.5, penalty="l2", solver="liblinear",
        max_iter=5000, random_state=SEED
    )),
])
cv = StratifiedKFold(n_splits=N_PROPENSITY_FOLDS, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(prop_model, X, y, cv=cv, method="predict_proba")[:, 1]

prop_auc = roc_auc_score(y, p_oof)
prop_logloss = log_loss(y, p_oof)

n_eval = len(eval_feat)
n_test = len(test_feat)
prior_eval = n_eval / (n_eval + n_test)
prior_test = n_test / (n_eval + n_test)

p_eval_as_test = np.clip(p_oof[:n_eval], 1e-4, 1 - 1e-4)
density_raw = (p_eval_as_test / (1 - p_eval_as_test)) * (prior_eval / prior_test)
density_clip = np.clip(density_raw, WEIGHT_CLIP_LOW, WEIGHT_CLIP_HIGH)
density_norm = density_clip / np.mean(density_clip)

eval_weights = eval_feat[["task_id"]].copy()
eval_weights["p_test_oof"] = p_eval_as_test
eval_weights["density_ratio_raw"] = density_raw
eval_weights["weight"] = density_norm
eval_weights["test_relevance_rank_pct"] = pd.Series(density_norm).rank(pct=True).to_numpy()
eval_weights["test_relevance_quartile"] = pd.qcut(
    pd.Series(density_norm).rank(method="first"),
    q=4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"]
).astype(str).to_numpy()

weight_ess = float((density_norm.sum() ** 2) / np.sum(density_norm ** 2))
eval_weights.to_csv(WORK_ROOT / "eval_test_propensity_weights.csv", index=False)

print("EXP-0 reproduction:")
print("  Eval-vs-Test CV AUC :", round(prop_auc, 6))
print("  Log loss            :", round(prop_logloss, 6))
print("  Weight ESS          :", round(weight_ess, 3), "/", len(eval_weights))
print("  Weight min/median/max:",
      round(float(density_norm.min()), 4),
      round(float(np.median(density_norm)), 4),
      round(float(density_norm.max()), 4))


# =====================================================================================
# SOURCE-ALIGNED PROXY-48 FEATURES
# Exact feature/tag rules copied from the source EDA notebook for audit comparability.
# These DO NOT replace EXP-0 features or official scoring.
# =====================================================================================
def source_eda_n_cells(grid):
    return int(np.prod(np.shape(grid)))

def source_eda_proxy_features(task_id, data):
    puzzle = data[task_id]
    train = puzzle["train"]
    input_sizes = [source_eda_n_cells(ex["input"]) for ex in train]
    output_sizes = [source_eda_n_cells(ex["output"]) for ex in train]
    ratios = [max(i, o) / max(min(i, o), 1) for i, o in zip(input_sizes, output_sizes)]
    colors = set()
    pixels = []
    for ex in train:
        for field in ["input", "output"]:
            a = np.asarray(ex[field])
            colors.update(np.unique(a).tolist())
            pixels.extend(a.reshape(-1).tolist())

    avg_input_size = float(np.mean(input_sizes))
    avg_ratio = float(np.mean(ratios))
    max_ratio = float(max(ratios))
    n_colors = len(colors)
    n_train = len(train)
    n_queries = len(puzzle["test"])
    is_dynamic = any(np.shape(ex["input"]) != np.shape(ex["output"]) for ex in train)
    is_rare = bool(6 in colors or 9 in colors)
    bg_density = pixels.count(0) / max(len(pixels), 1)

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )
    if n_train >= 6 and tier < 3:
        fine *= 0.7

    tags = [f"tier{tier}"]
    if tier == 1 and (not is_dynamic) and max_ratio == 1 and n_colors <= 5:
        tags.append("easy_static")
    if tier == 1 and fine <= 0.22:
        tags.append("easy")
    if tier == 1 and fine > 0.45:
        tags.append("hard_tier1")
    if is_dynamic:
        tags.append("dynamic")
    if is_rare:
        tags.append("rare")
    if n_queries > 1:
        tags.append("multi_query")
    if max_ratio > 5:
        tags.append("high_ratio")
    if max_ratio > 30:
        tags.append("extreme_ratio")
    if n_colors >= 8:
        tags.append("many_colors")
    if avg_input_size >= 500:
        tags.append("large")
    if bg_density >= 0.70:
        tags.append("sparse_bg")
    if bg_density <= 0.30:
        tags.append("dense_non_bg")
    if n_train <= 2:
        tags.append("low_train")

    return {
        "task_id": task_id,
        "proxy_tier": int(tier),
        "proxy_fine": float(fine),
        "proxy_avg_input_size": avg_input_size,
        "proxy_max_ratio": max_ratio,
        "proxy_avg_ratio": avg_ratio,
        "proxy_n_colors": int(n_colors),
        "proxy_dynamic": int(is_dynamic),
        "proxy_rare": int(is_rare),
        "proxy_n_train": int(n_train),
        "proxy_n_queries": int(n_queries),
        "proxy_bg_density": float(bg_density),
        "proxy_easy_static": int("easy_static" in tags),
        "proxy_hard_tier1": int("hard_tier1" in tags),
        "proxy_high_ratio": int("high_ratio" in tags),
        "proxy_extreme_ratio": int("extreme_ratio" in tags),
        "proxy_tags": "|".join(tags),
    }

proxy_source_feat = pd.DataFrame([
    source_eda_proxy_features(k, eval_challenges) for k in eval_challenges
]).sort_values("task_id").reset_index(drop=True)
proxy_source_feat.to_csv(WORK_ROOT / "source_eda_features_eval.csv", index=False)

proxy48_source = proxy_source_feat[proxy_source_feat["task_id"].isin(PROXY48_TASK_IDS)].copy()
proxy48_order = {k: i for i, k in enumerate(PROXY48_TASK_IDS)}
proxy48_source["_order"] = proxy48_source["task_id"].map(proxy48_order)
proxy48_source = proxy48_source.sort_values("_order").drop(columns="_order").reset_index(drop=True)
proxy48_source.to_csv(WORK_ROOT / "proxy48_source_eda.csv", index=False)

proxy_source_counts = {
    "total": int(len(proxy48_source)),
    "tier1": int((proxy48_source["proxy_tier"] == 1).sum()),
    "tier2": int((proxy48_source["proxy_tier"] == 2).sum()),
    "tier3": int((proxy48_source["proxy_tier"] == 3).sum()),
    "easy_static": int(proxy48_source["proxy_easy_static"].sum()),
    "rare": int(proxy48_source["proxy_rare"].sum()),
    "multi_query": int((proxy48_source["proxy_n_queries"] > 1).sum()),
    "dynamic": int(proxy48_source["proxy_dynamic"].sum()),
    "hard_tier1": int(proxy48_source["proxy_hard_tier1"].sum()),
    "high_ratio": int(proxy48_source["proxy_high_ratio"].sum()),
    "extreme_ratio": int(proxy48_source["proxy_extreme_ratio"].sum()),
}
if proxy_source_counts != PROXY48_SOURCE_COUNTS:
    raise RuntimeError(
        "Proxy-48 source-EDA profile drift detected. "
        f"expected={PROXY48_SOURCE_COUNTS}, actual={proxy_source_counts}"
    )
with open(WORK_ROOT / "proxy48_source_distribution.json", "w", encoding="utf-8") as f:
    json.dump(proxy_source_counts, f, indent=2)
print("Proxy-48 source-EDA contract: PASS", proxy_source_counts)


EXP-0 reproduction:
  Eval-vs-Test CV AUC : 0.888229
  Log loss            : 0.42451
  Weight ESS          : 32.57 / 120
  Weight min/median/max: 0.2285 0.2285 5.7127
Proxy-48 source-EDA contract: PASS {'total': 48, 'tier1': 32, 'tier2': 12, 'tier3': 4, 'easy_static': 16, 'rare': 30, 'multi_query': 8, 'dynamic': 8, 'hard_tier1': 2, 'high_ratio': 5, 'extreme_ratio': 4}


In [4]:
# =====================================================================================
# CELL 4 — TOKENIZER + SOURCE-ALIGNED AGENTIC PROMPT + QWEN TOOL TEMPLATE + ACTIVE TASKS
# =====================================================================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True, trust_remote_code=False)
print("Tokenizer:", tokenizer.__class__.__name__, "| vocab:", len(tokenizer))

# OpenAI-style schema; Qwen3.8's official chat template serializes it into its XML tool-call contract.
PYTHON_TOOL = {
    "type": "function",
    "function": {
        "name": "python",
        "description": "Execute Python in a persistent per-puzzle REPL and return stdout or a concise error.",
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "Python code to execute"}},
            "required": ["code"],
            "additionalProperties": False,
        },
    },
}
TOOL_SCHEMA_SHA256 = stable_json_sha256(PYTHON_TOOL)

SOURCE_SYSTEM_PROMPT = '# Tools\n\n## python\n\nUse this tool to execute Python code in your chain of thought. \n\nWhen you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. The environment supports common third party libraries like numpy, scipy, shapely, networkx, scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver. python will respond with the output of the execution or time out after 120.0 seconds. \n\n### Usage guideline\n\n1. Use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions.\n2. Prefer to write reusable functions. These will persist in the stateful Jupyter notebook and can be referenced in later code snippets without having to define them again.'
SOURCE_AGENTIC_PROMPT_TEMPLATE = 'You are an expert ARC AGI puzzle solver. Please solve the puzzle given below.\n\n# Background information about ARC AGI puzzles\n\nEach task consists of a small number of demonstration examples (3.3 on average), and a small number of test examples (1 - 3).\n\nEach example consists of an “input grid” and an “output grid”. Each “grid” is a literal grid of colored squares There are 10 unique colors. A grid can be any height or width between 1x1 and 30x30, inclusive.\n\nThe color code is as follows:\n\n0 - black\n1 - navy blue\n2 - red\n3 - green\n4 - yellow\n5 - gray\n6 - pink\n7 - orange\n8 - sky blue\n9 - maroon\n\nWhen solving an evaluation task, a test-taker has access to the training examples for the task (both the input and output grids), as well as the input grid of the test examples for the task. The test-taker must construct on its own the output grid corresponding to the input grid of each test example. “Constructing the output grid” is done entirely from scratch, meaning that the test-taker must decide what the height and width of the output grid should be, what colors they should place on the grid, and where. The task is successfully solved if the test-taker can produce the exact correct answer on all test examples for the task (binary measure of success).\n\nThe mapping from integers (0-9) to color names is arbitrary and should be treated as categorical labels; the transformation rule never relies on the specific integer–color pairing.\n\nARC AGI puzzle solutions feel "natural" to humans. Natural rules tend to build on visual priors and small-number counting. They are visually immediate or prominent for humans. That is why humans from all educational backgrounds are able to solve these puzzles. Rules that depend on non-visual considerations (e.g. mathematics that\'s more advanced than small number counting) or inelegant mechanics (e.g. tie-breakers, complex conditional rules, special exceptions) are considered "unnatural."  \n\nThis doesn\'t mean puzzles have to be simple or easy. Common patterns that puzzles use to increase difficulty are:  \n\n- **Rule composition:** multiple simple, natural rules are applied in sequence, with one transformation setting up the next.\n- **Contextual application:** a simple if/else based on a visually clear feature.\n- **In-grid symbol definition:** certain shapes or patterns act as symbols whose meaning is established using the training examples and which control some aspect of the transformation. The meaning can even vary across training examples.\n- **Iterated steps:** a simple action is applied repeatedly (as in assembling pieces of a jigsaw), where each step constrains the next. Repetition is restricted to a small number.\n- **Approximate features**: Features that are easily detected by humans, but hard to detect using naive algorithms. Examples are slightly broken symmetry, slightly broken continuity, off-center symmetry axes, objects defined via 8-connectivity etc.\n- **Non-locality:** the solution requires applying changes to one region, with the changes depending on features in a different, distant region.\n\n# Puzzle to solve\n\n{{puzzle}}\n\n# Task aids\n\nYou may execute code in the Jupyter notebook environment.\n\nVariables available in the Jupyter notebook:\n\n- `puzzle`: A Python `dict` containing the training and test grids. The key "train" contains the training examples in a list. Similarly, the key "test" contains the test examples as a list. Each training/test example is a dict containing keys "input" and "output". The values are the grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.\n\nYour very first code execution should be `puzzle.keys()` to check data access. \n\nDon\'t print the `puzzle` dict, because it prints out the grids that are already present in this user message and consumes valuable token budget.\n\n# Your task\n\nPlease solve the puzzle. Solving the puzzle means constructing a function `solution(grid: list[list[int]])` that implements the transformation from input grid to output grid shown in the training examples. The solution must work on all training examples.\n\n```\nfor idx, example in enumerate(puzzle["train"]):\n    assert solution(example["input"]) == example["output"], f"Train example {idx}: mismatch"\n```\n\nThe solution must generalize to the test input(s). Solutions that cheat by hardcoding the training example outputs are not acceptable.\n\n# Output format (STRICT)\n\nYour response must contain a single Python code block fenced by ```python ... ```. Must include a `solution` function. Must be self-contained i.e. include all necessary imports and helper functions.'

# LOCAL adaptation: source advertises a broad dependency list and 120s IPython timeout.
# The portability audit recommends a smaller P0 stack. Probe the actual current Python
# environment and advertise only approved libraries that really import.
import subprocess as _piv_subprocess
P0_LIBRARY_PROBES = [
    ("numpy","numpy"), ("scipy","scipy"), ("scikit-image","skimage"),
    ("networkx","networkx"), ("Pillow","PIL"),
]
AVAILABLE_P0_LIBRARIES=[]
P0_LIBRARY_PROBE_RESULTS={}
for _label,_module in P0_LIBRARY_PROBES:
    try:
        _cp=_piv_subprocess.run(
            [sys.executable,"-c",f"import {_module}"],
            text=True,capture_output=True,timeout=8,
        )
        _ok=(_cp.returncode==0)
        P0_LIBRARY_PROBE_RESULTS[_label]={"ok":_ok,"stderr":(_cp.stderr or "")[-500:]}
        if _ok: AVAILABLE_P0_LIBRARIES.append(_label)
    except Exception as _e:
        P0_LIBRARY_PROBE_RESULTS[_label]={"ok":False,"stderr":f"{type(_e).__name__}: {_e}"}

_source_lib_sentence=(
    "The environment supports common third party libraries like numpy, scipy, shapely, networkx, "
    "scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver."
)
_local_lib_sentence=(
    "For this LOCAL P0 port, approved third party libraries verified in the current environment are: "
    + (", ".join(AVAILABLE_P0_LIBRARIES) if AVAILABLE_P0_LIBRARIES else "none beyond the Python standard library")
    + ". Do not assume optional source libraries such as OR-Tools, z3-solver, python-constraint, or shapely are available."
)
SYSTEM_PROMPT = SOURCE_SYSTEM_PROMPT.replace(_source_lib_sentence,_local_lib_sentence).replace(
    "time out after 120.0 seconds",
    f"time out after {TOOL_TIMEOUT_SEC:.1f} seconds",
)
SYSTEM_PROMPT += (
    f"\n\n### Local P0 compute budget\n"
    f"You may execute Python at most {MAX_TOOL_CALLS} times in this task. "
    "Use the calls deliberately. When the final Python call has been used, do not request another tool call; "
    "return the self-contained final `solution(grid)` code block on the next assistant turn."
)

def grid_lines(grid):
    return "\n".join(" ".join(str(int(x)) for x in row) for row in grid)

def puzzle_to_source_text(task):
    parts = ["## Training examples"]
    for i, ex in enumerate(task["train"]):
        parts += [f"### Training example {i}", "Input:", grid_lines(ex["input"]), "Output:", grid_lines(ex["output"])]
    parts.append("## Tests" if len(task["test"]) > 1 else "## Test")
    for i, ex in enumerate(task["test"]):
        if len(task["test"]) > 1:
            parts.append(f"### Test {i}")
        parts += ["Input:", grid_lines(ex["input"])]
    return "\n\n".join(parts)

def build_agent_user_prompt(task):
    # Keep the source prompt intact except for two necessary local substitutions:
    # 1) serialize the current puzzle; 2) clarify that test outputs are absent from the preloaded variable.
    prompt = SOURCE_AGENTIC_PROMPT_TEMPLATE.replace("{{puzzle}}", puzzle_to_source_text(task))
    prompt = prompt.replace(
        'Each training/test example is a dict containing keys "input" and "output". '
        'The values are the grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.',
        'Each training example is a dict containing keys "input" and "output"; each test example contains only "input". '
        'The values are grids expressed as list of lists of integers (0 - 9). '
        'Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.'
    )
    return prompt

SOURCE_AGENTIC_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_AGENTIC_PROMPT_TEMPLATE.encode("utf-8")).hexdigest()
SOURCE_SYSTEM_PROMPT_LOCAL_SHA256 = hashlib.sha256(SOURCE_SYSTEM_PROMPT.encode("utf-8")).hexdigest()
with open(WORK_ROOT/"p0_library_probe.json","w") as _f:
    json.dump({"available":AVAILABLE_P0_LIBRARIES,"results":P0_LIBRARY_PROBE_RESULTS},_f,indent=2)
print("P0 approved libraries available:",AVAILABLE_P0_LIBRARIES)

def render_agent_chat(messages):
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
        tools=[PYTHON_TOOL],
        enable_thinking=True,
        reasoning_effort="xhigh",
        preserve_thinking=True,
    )
    try:
        return tokenizer.apply_chat_template(messages, **kwargs), "qwen_tools+xhigh+preserve_thinking"
    except TypeError as exc:
        # Preserve the causal mechanism if this repacked tokenizer predates the explicit preserve kwarg.
        kwargs.pop("preserve_thinking", None)
        rendered = tokenizer.apply_chat_template(messages, **kwargs)
        return rendered, f"qwen_tools+xhigh(template-preserve-default); fallback={type(exc).__name__}"

# Frozen task selection: same Proxy-48; SMOKE is a deterministic subset of it.
def choose_smoke8_from_proxy():
    """v1.6: stratified to mirror Proxy-48 (tier1 32/48, tier2 12/48, tier3 4/48 -> 5/2/1), deterministic in
    Proxy-48 order. The previous first-match selector took all four tier-3 tasks and biased the feasibility gate."""
    f = proxy_source_feat.set_index("task_id").loc[PROXY48_TASK_IDS].copy()
    chosen=[]
    def add(x):
        if x is not None and x not in chosen and len(chosen)<8: chosen.append(x)
    def first(mask):
        for t in f.index[mask.to_numpy()]:
            if t not in chosen: return t
        return None
    t1=(f["proxy_tier"]==1); t2=(f["proxy_tier"]==2); t3=(f["proxy_tier"]==3)
    add(first(t3))
    add(first(t2)); add(first(t2))
    add(first(t1 & (f["proxy_easy_static"]==1)))
    add(first(t1 & (f["proxy_hard_tier1"]==1)))
    add(first(t1 & (f["proxy_n_queries"]>1)))
    add(first(t1 & (f["proxy_dynamic"]==1)))
    while len(chosen)<8 and first(t1) is not None: add(first(t1))
    for x in PROXY48_TASK_IDS:
        if len(chosen)>=8: break
        add(x)
    return chosen[:8]

if TASK_SET == "PROXY48": ACTIVE_TASK_IDS = list(PROXY48_TASK_IDS)
elif TASK_SET == "SMOKE8": ACTIVE_TASK_IDS = choose_smoke8_from_proxy()
else: raise RuntimeError(f"Unknown TASK_SET={TASK_SET}")
ACTIVE_TASK_SET=set(ACTIVE_TASK_IDS)
if len(ACTIVE_TASK_IDS)!=ACTIVE_TASK_LIMIT or len(ACTIVE_TASK_SET)!=ACTIVE_TASK_LIMIT:
    raise RuntimeError("Active task count/uniqueness mismatch")
if not ACTIVE_TASK_SET.issubset(set(PROXY48_TASK_IDS)):
    raise RuntimeError("Active set escaped frozen Proxy-48")

prompt_records=[]
for task_id, task in eval_challenges.items():
    user_prompt=build_agent_user_prompt(task)
    messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_prompt}]
    rendered, method=render_agent_chat(messages)
    prompt_records.append({
        "task_id":task_id, "user_prompt":user_prompt, "initial_messages":messages,
        "prompt":rendered, "prompt_sha256":hashlib.sha256(rendered.encode()).hexdigest(),
        "prompt_tokens":len(tokenizer.encode(rendered, add_special_tokens=False)),
        "n_train":len(task["train"]), "n_test":len(task["test"]), "render_method":method,
    })
prompt_df=pd.DataFrame([{k:v for k,v in r.items() if k not in {"prompt","initial_messages","user_prompt"}} for r in prompt_records])
prompt_df.to_csv(WORK_ROOT/"prompt_stats.csv", index=False)
PROMPT_RECORD_MAP={r["task_id"]:r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET}

chat_template=getattr(tokenizer,"chat_template","") or ""
template_sha=hashlib.sha256(chat_template.encode()).hexdigest()
if "<tool_call>" not in chat_template or "tool_response" not in chat_template:
    raise RuntimeError("Mounted Qwen chat template does not expose the required tool-call/tool-response contract.")

# Validate the *actual mounted tokenizer* on the multi-turn structure PIV will use.
_transport_probe_messages = [
    {"role":"system","content":SYSTEM_PROMPT},
    {"role":"user","content":"transport probe"},
    {"role":"assistant","content":"","reasoning_content":"PIV_REASONING_SENTINEL",
     "tool_calls":[{"type":"function","function":{"name":"python","arguments":{"code":"x=1\nx+1"}}}]},
    {"role":"tool","content":"2"},
]
_transport_rendered, _transport_method = render_agent_chat(_transport_probe_messages)
for _required in ("PIV_REASONING_SENTINEL", "<tool_call>", "<function=python>", "<parameter=code>", "<tool_response>", "2"):
    if _required not in _transport_rendered:
        raise RuntimeError(f"Qwen multi-turn tool-history rendering contract failed: missing {_required!r}")
if not _transport_rendered.rstrip().endswith("<think>"):
    raise RuntimeError("Qwen tool-history render did not reopen thinking for the next assistant turn.")
QWEN_TOOL_HISTORY_TRANSPORT_SHA256 = hashlib.sha256(_transport_rendered.encode("utf-8")).hexdigest()
with open(WORK_ROOT/"chat_template_fingerprint.txt","w") as f:
    f.write(f"sha256={template_sha}\ntool_schema_sha256={TOOL_SCHEMA_SHA256}\n")

profile_task_table=eval_feat[eval_feat["task_id"].isin(ACTIVE_TASK_IDS)].copy()
profile_task_table=profile_task_table.merge(proxy_source_feat,on="task_id",how="left")
profile_task_table["prompt_tokens_piv_initial"]=profile_task_table["task_id"].map(prompt_df.set_index("task_id")["prompt_tokens"])
profile_task_table["proxy_order"]=profile_task_table["task_id"].map(proxy48_order)
profile_task_table=profile_task_table.sort_values("proxy_order").reset_index(drop=True)
profile_task_table.to_csv(WORK_ROOT/"active_task_selection.csv",index=False)

SOURCE_ALIGNMENT={
    "experiment":EXPERIMENT_ID,
    "piv_source_repo":PIV_SOURCE_REPO,
    "piv_source_commit":PIV_SOURCE_COMMIT,
    "piv_source_files":PIV_SOURCE_FILES,
    "external_sources":PIV_EXTERNAL_SOURCES,
    "proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,
    "source_file_sha256":SOURCE_FILE_SHA256,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
    "chat_template_sha256":template_sha,
    "qwen_tool_history_transport_sha256":QWEN_TOOL_HISTORY_TRANSPORT_SHA256,
    "source_agentic_prompt_local_sha256":SOURCE_AGENTIC_PROMPT_LOCAL_SHA256,
    "source_system_prompt_local_sha256":SOURCE_SYSTEM_PROMPT_LOCAL_SHA256,
    "source_invariants":[
        "one persistent Python REPL per lineage",
        "puzzle preloaded without test outputs",
        "tool errors returned to model",
        "self-contained solution(grid) final artifact",
        "fresh independent all-train exact verification",
        "test grids produced by verified program execution",
    ],
    "local_adaptations":[
        "direct vLLM + Qwen native tool chat template; no OpenAI Responses server",
        "lightweight subprocess worker instead of Docker/IPyBox",
        "stricter grid/domain/time/file-system safety",
        "bounded 6-execution P0 tool budget and context guard",
        "source-style max_retries=2 reliability retries share the same total token/tool/turn budgets",
        "per-turn ceiling = remaining total P0 budget (12288) bounded by context headroom; total generated-token cap 12288",
    ],
}
with open(WORK_ROOT/"SOURCE_ALIGNMENT.json","w") as f: json.dump(SOURCE_ALIGNMENT,f,indent=2)
print("Chat template/tool contract PASS | SHA:",template_sha)
print("ACTIVE TASKS:",len(ACTIVE_TASK_IDS),TASK_SET)
display(prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].describe().to_frame("initial_prompt_tokens").round(1))


Tokenizer: Qwen2TokenizerFast | vocab: 248077
P0 approved libraries available: ['numpy', 'scipy', 'scikit-image', 'networkx', 'Pillow']
Chat template/tool contract PASS | SHA: c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041
ACTIVE TASKS: 48 PROXY48


,initial_prompt_tokens
count,48.0
mean,6537.4
std,2438.0
min,3057.0
25%,4794.5
50%,6270.5
75%,7929.8
max,13431.0


In [5]:
# =====================================================================================
# CELL 5 — PIV CONTEXT / COMPUTE SAFETY (preserve history; do not add P3 compaction in P0)
# =====================================================================================
active_prompt_tokens=prompt_df[prompt_df.task_id.isin(ACTIVE_TASK_IDS)]["prompt_tokens"].to_numpy()
max_initial_prompt=int(active_prompt_tokens.max())
CONTEXT_SAFETY_TOKENS=512
MAX_RENDERED_INPUT_TOKENS=MAX_MODEL_LEN-PER_TURN_MAX_TOKENS-CONTEXT_SAFETY_TOKENS
if max_initial_prompt > MAX_RENDERED_INPUT_TOKENS:
    raise RuntimeError(f"Initial PIV prompt exceeds context: {max_initial_prompt} > {MAX_RENDERED_INPUT_TOKENS}")
MAX_TOOL_OUTPUT_CHARS=4000
MAX_TOOL_RAW_ARTIFACT_CHARS=100000
MAX_CODE_CHARS=30000
print("PIV CONTEXT CONFIG")
print("  max_model_len           :",MAX_MODEL_LEN)
print("  max initial prompt      :",max_initial_prompt)
print("  per-turn max generation:",PER_TURN_MAX_TOKENS)
print("  total generated cap     :",TOTAL_GENERATED_TOKEN_CAP)
print("  max rendered input      :",MAX_RENDERED_INPUT_TOKENS)
print("  max tool output to model:",MAX_TOOL_OUTPUT_CHARS,"chars")
print("  history policy          : PRESERVE (P0); no P3 compact-state intervention")


PIV CONTEXT CONFIG
  max_model_len           : 29696
  max initial prompt      : 13431
  per-turn max generation: 12288
  total generated cap     : 12288
  max rendered input      : 16896
  max tool output to model: 4000 chars
  history policy          : PRESERVE (P0); no P3 compact-state intervention


In [6]:
# =====================================================================================
# CELL 6 — QWEN TOOL XML PARSER + PERSISTENT LOCAL REPL + FRESH EXACT VERIFIER + SELF-TESTS
# =====================================================================================
import ast as py_ast
import re, subprocess, selectors, tempfile, uuid

DANGEROUS_IMPORT_ROOTS={"os","subprocess","pathlib","shutil","socket","requests","urllib","http","ftplib","paramiko","builtins","importlib","inspect"}
DANGEROUS_CALL_NAMES={"open","exec","eval","compile","__import__","input","breakpoint","globals","locals","vars","getattr","setattr","delattr","exit","quit"}
DANGEROUS_DUNDER_ATTRS={"__subclasses__","__globals__","__builtins__","__code__","__closure__","__getattribute__"}
ALLOWED_SYS_ATTRS={"setrecursionlimit","getrecursionlimit","maxsize"}

def canonical_grid(grid):
    if isinstance(grid,np.ndarray): grid=grid.tolist()
    if not is_rectangular_grid(grid): return None
    return tuple(tuple(int(x) for x in row) for row in grid)
def grids_equal(a,b):
    ca,cb=canonical_grid(a),canonical_grid(b)
    return ca is not None and cb is not None and ca==cb

def code_defines_solution(code):
    try: tree=py_ast.parse(code)
    except Exception: return False
    return any(isinstance(n,(py_ast.FunctionDef,py_ast.AsyncFunctionDef)) and n.name=="solution" for n in tree.body)

def extract_last_solution_code(text):
    blocks=re.findall(r"```(?:python|py)?\s*\n?(.*?)```", text or "", flags=re.I|re.S)
    for code in reversed(blocks):
        code=code.strip()
        if code_defines_solution(code): return code
    return None

def split_qwen_reasoning(text, action_pos=None):
    text=text or ""
    prefix=text if action_pos is None else text[:action_pos]
    if "</think>" in prefix:
        a,b=prefix.rsplit("</think>",1)
        return a.replace("<think>","",1).strip(), b.strip(), "think_close"
    # Generation prompt already opens <think>; without a close, pre-action text is reasoning.
    return prefix.replace("<think>","",1).strip(), "", "implicit_open_think"

def _action_region(text):
    """Prefer the model-action region after the final </think>; fall back to all text if no close exists."""
    text=text or ""
    if "</think>" in text:
        pos=text.rfind("</think>")+len("</think>")
        return text[pos:],pos
    return text,0

def parse_tool_calls(text):
    text=text or ""
    region,offset=_action_region(text)
    calls=[]
    pattern=r"<tool_call>\s*<function=([^>\s]+)>\s*(.*?)</function>\s*</tool_call>"
    for m in re.finditer(pattern,region,re.S):
        name=m.group(1).strip(); body=m.group(2); args={}
        for p in re.finditer(r"<parameter=([^>\s]+)>\s*(.*?)\s*</parameter>",body,re.S):
            val=p.group(2)
            # v1.6: unwrap a fenced ```python payload; the fence itself is not Python.
            _fm=re.fullmatch(r"```(?:python|py)?[ \t]*\n?(.*?)\n?```",val.strip(),flags=re.S|re.I)
            if _fm: val=_fm.group(1)
            args[p.group(1).strip()]=val
        calls.append({"name":name,"arguments":args,"start":offset+m.start(),"end":offset+m.end(),"raw":m.group(0)})
    return calls

def parse_agent_turn(text):
    text=text or ""
    tcs=parse_tool_calls(text)
    if tcs:
        reasoning,content,rmode=split_qwen_reasoning(text,tcs[0]["start"])
        suffix=text[tcs[-1]["end"]:].strip()
        suffix=re.sub(r"(?:<\|im_end\|>|<\|endoftext\|>)\s*$","",suffix).strip()
        return {"kind":"tool","tool_calls":tcs,"tool_name":tcs[0]["name"],"tool_args":tcs[0]["arguments"],
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":suffix,"final_code":None}
    action_text,_=_action_region(text)
    if "<tool_call>" in action_text:   # v1.6: prose merely mentioning <function= is not a malformed call
        reasoning,content,rmode=split_qwen_reasoning(text,None)
        return {"kind":"malformed_tool","tool_calls":[],"tool_name":None,"tool_args":{},
                "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
                "suffix_after_tool":"","final_code":None}
    code=extract_last_solution_code(action_text)
    reasoning,content,rmode=split_qwen_reasoning(text,None)
    return {"kind":"final" if code else "text","tool_calls":[],"tool_name":None,"tool_args":{},
            "reasoning_content":reasoning,"content":content,"reasoning_mode":rmode,
            "suffix_after_tool":"","final_code":code}

def generation_finish_reason_ok(finish_reason):
    # Mirrors the source client's rule that a truncated response is not a valid turn result.
    return str(finish_reason).lower() == "stop"

# v1.6: tool output is inserted verbatim into the next prompt. Neutralize chat-control strings the model's
# own code could print so they cannot open/close turns or tool blocks.
_CHAT_CONTROL_PATTERNS=[("<|","< |"),("<tool_response>","< tool_response>"),("</tool_response>","< /tool_response>"),
                        ("<tool_call>","< tool_call>"),("</tool_call>","< /tool_call>"),
                        ("<think>","< think>"),("</think>","< /think>")]
def sanitize_tool_output(text):
    text=str(text or "")
    for a,b in _CHAT_CONTROL_PATTERNS: text=text.replace(a,b)
    return text

def salvage_final_code_on_length(text, parsed):
    """v1.6 LOCAL: a length-capped final turn whose action region (after </think>) already contains a CLOSED,
    ast-parseable ```python block defining solution() is demonstrably complete; retrying it would only burn
    budget. Drafts inside an unclosed <think> are never salvaged."""
    if parsed.get("kind")!="final" or parsed.get("reasoning_mode")!="think_close": return None
    return parsed.get("final_code")

def guard_python_code(code, *, final=False):
    if not isinstance(code,str) or not code.strip(): return False,"empty_code"
    if len(code)>MAX_CODE_CHARS: return False,"code_too_long"
    try: tree=py_ast.parse(code)
    except SyntaxError as e: return False,f"syntax_error:{e.msg}"

    # `sys.setrecursionlimit(...)` is common in DFS/flood-fill ARC solutions.
    # Permit only a narrow sys surface; block module-table/process introspection.
    sys_aliases=set()
    for _n in py_ast.walk(tree):
        if isinstance(_n,py_ast.Import):
            for _a in _n.names:
                if _a.name=="sys": sys_aliases.add(_a.asname or "sys")
        elif isinstance(_n,py_ast.ImportFrom) and (_n.module or "")=="sys":
            imported={_a.name for _a in _n.names}
            if not imported.issubset(ALLOWED_SYS_ATTRS):
                return False,f"blocked_sys_importfrom:{sorted(imported)}"

    for node in py_ast.walk(tree):
        if isinstance(node,py_ast.Name) and node.id=="__builtins__":
            return False,"blocked_builtins_escape"
        if isinstance(node,py_ast.Attribute) and node.attr in DANGEROUS_DUNDER_ATTRS:
            return False,f"blocked_dunder_attribute:{node.attr}"
        if isinstance(node,py_ast.Attribute) and isinstance(node.value,py_ast.Name) and node.value.id in sys_aliases:
            if node.attr not in ALLOWED_SYS_ATTRS:
                return False,f"blocked_sys_attribute:{node.attr}"
        if isinstance(node,py_ast.Constant) and isinstance(node.value,str) and "/kaggle" in node.value.lower():
            return False,"blocked_host_path_literal"
        if isinstance(node,(py_ast.Import,py_ast.ImportFrom)):
            names=[a.name.split('.')[0] for a in node.names] if isinstance(node,py_ast.Import) else [(node.module or '').split('.')[0]]
            if any(x in DANGEROUS_IMPORT_ROOTS for x in names): return False,f"blocked_import:{names}"
        if isinstance(node,py_ast.Call):
            if isinstance(node.func,py_ast.Name) and node.func.id in DANGEROUS_CALL_NAMES:
                return False,f"blocked_call:{node.func.id}"
            if isinstance(node.func,py_ast.Attribute) and node.func.attr in DANGEROUS_DUNDER_ATTRS:
                return False,f"blocked_dunder_call:{node.func.attr}"
        if isinstance(node,py_ast.Raise):
            exc=node.exc
            if isinstance(exc,py_ast.Name) and exc.id=="SystemExit":
                return False,"blocked_system_exit"
            if isinstance(exc,py_ast.Call) and isinstance(exc.func,py_ast.Name) and exc.func.id=="SystemExit":
                return False,"blocked_system_exit"
        if isinstance(node,(py_ast.Assign,py_ast.AnnAssign,py_ast.AugAssign,py_ast.NamedExpr)):
            targets=[]
            if isinstance(node,py_ast.Assign): targets=node.targets
            else: targets=[node.target]
            def rooted_at_puzzle(t):
                while isinstance(t,(py_ast.Subscript,py_ast.Attribute)):
                    t=t.value
                return isinstance(t,py_ast.Name) and t.id=="puzzle"
            for t in targets:
                if rooted_at_puzzle(t): return False,"protected_puzzle_assignment"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Attribute):
            base=node.func.value
            while isinstance(base,(py_ast.Subscript,py_ast.Attribute)): base=base.value
            if isinstance(base,py_ast.Name) and base.id=="puzzle" and node.func.attr in {"update","clear","pop","popitem","setdefault","append","extend","insert","remove"}:
                return False,"protected_puzzle_mutation"
        if isinstance(node,py_ast.Expr) and isinstance(node.value,py_ast.Name) and node.value.id=="puzzle":
            return False,"blocked_whole_puzzle_display"
        if isinstance(node,py_ast.Call) and isinstance(node.func,py_ast.Name) and node.func.id=="print":
            if any(isinstance(a,py_ast.Name) and a.id=="puzzle" for a in node.args):
                return False,"blocked_whole_puzzle_print"
    if final and not code_defines_solution(code): return False,"missing_solution_function"
    return True,"ok"

def compact_tool_output(text,limit=MAX_TOOL_OUTPUT_CHARS):
    text=str(text or "")
    if len(text)<=limit: return text
    keep=max(1,(limit-120)//2)
    return text[:keep]+f"\n...[tool output truncated; original chars={len(text)}]...\n"+text[-keep:]

WORKER_SCRIPT = r"""
import sys,json,traceback,io,contextlib,copy,ast,os
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV sandbox blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV sandbox blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
original_puzzle=None
for line in sys.stdin:
    try:
        msg=json.loads(line); op=msg.get("op")
        if op=="init":
            original_puzzle=copy.deepcopy(msg["puzzle"])
            ns={"__name__":"__main__","puzzle":copy.deepcopy(original_puzzle)}
            out={"ok":True,"text":"REPL initialized."}
        elif op=="exec":
            # Restore the protected puzzle binding before every cell. Other REPL state persists.
            ns["puzzle"]=copy.deepcopy(original_puzzle)
            buf=io.StringIO()
            try:
                with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                    # Jupyter-like semantics: execute statements, then display repr(last expression).
                    tree=ast.parse(msg["code"],mode="exec")
                    if tree.body and isinstance(tree.body[-1],ast.Expr):
                        prefix=ast.Module(body=tree.body[:-1],type_ignores=[])
                        if prefix.body: exec(compile(prefix,"<piv-repl>","exec"),ns,ns)
                        value=eval(compile(ast.Expression(tree.body[-1].value),"<piv-repl>","eval"),ns,ns)
                        if value is not None: print(repr(value))
                    else:
                        exec(compile(tree,"<piv-repl>","exec"),ns,ns)
                out={"ok":True,"text":buf.getvalue() or "Code executed successfully. No output was returned."}
            except BaseException:
                out={"ok":False,"text":buf.getvalue()+traceback.format_exc(limit=8)}
        elif op=="close":
            print(json.dumps({"ok":True,"text":"closing"}),flush=True); break
        else: out={"ok":False,"text":"Unknown worker op"}
    except BaseException:
        out={"ok":False,"text":traceback.format_exc(limit=8)}
    print(json.dumps(out,ensure_ascii=False),flush=True)
"""

class PersistentPythonWorker:
    def __init__(self,puzzle,timeout_sec=10.0):
        self.puzzle=puzzle; self.timeout_sec=float(timeout_sec); self.proc=None; self.successful_cells=[]; self.restart_count=0
        self.start()
    def start(self):
        self.close(silent=True)
        self.proc=subprocess.Popen([sys.executable,"-u","-c",WORKER_SCRIPT],stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                                   stderr=subprocess.DEVNULL,text=True,bufsize=1,cwd=tempfile.mkdtemp(prefix="piv_repl_"))
        r=self._request({"op":"init","puzzle":self.puzzle},timeout=max(2.0,self.timeout_sec))
        if not r.get("ok"): raise RuntimeError(f"REPL init failed: {r}")
        # Replay only previously successful bounded cells after a timeout/crash.
        if self.successful_cells:
            cells=list(self.successful_cells); self.successful_cells=[]
            for c in cells:
                rr=self.execute(c,record=True,replay=True)
                if not rr.get("ok"): break
    def _request(self,msg,timeout):
        if self.proc is None or self.proc.poll() is not None: raise RuntimeError("REPL process is not alive")
        self.proc.stdin.write(json.dumps(msg,ensure_ascii=False)+"\n"); self.proc.stdin.flush()
        sel=selectors.DefaultSelector(); sel.register(self.proc.stdout,selectors.EVENT_READ)
        events=sel.select(timeout); sel.close()
        if not events: raise TimeoutError(f"REPL tool exceeded {timeout}s")
        line=self.proc.stdout.readline()
        if not line: raise RuntimeError("REPL worker closed stdout")
        return json.loads(line)
    def execute(self,code,record=True,replay=False):
        ok,reason=guard_python_code(code,final=False)
        if not ok: return {"ok":False,"text":f"Tool code rejected by local safety guard: {reason}","guard_rejected":True,"restarted":False}
        try:
            r=self._request({"op":"exec","code":code},self.timeout_sec); r["restarted"]=False
            if r.get("ok") and record: self.successful_cells.append(code)
            return r
        except TimeoutError:
            self.restart_count+=1
            old=list(self.successful_cells)
            self.successful_cells=old
            try: self.start()
            except Exception: pass
            return {"ok":False,"text":f"Code execution terminated after {self.timeout_sec}s. Persistent worker was restarted; prior successful cells were replayed when possible.","timeout":True,"restarted":True}
        except Exception as e:
            self.restart_count+=1
            restarted=False
            try:
                self.start(); restarted=True
            except Exception:
                restarted=False
            return {"ok":False,"text":f"REPL infrastructure error: {type(e).__name__}: {e}. "
                                      f"Worker restart attempted; prior successful cells replayed when possible.",
                    "infrastructure_error":True,"restarted":restarted}
    def close(self,silent=False):
        p=getattr(self,"proc",None)
        if p is not None and p.poll() is None:
            try: p.terminate(); p.wait(timeout=1)
            except Exception:
                try: p.kill()
                except Exception: pass
        self.proc=None

VERIFIER_SCRIPT = r"""
import sys,json,copy,traceback,numbers,os
payload=json.loads(sys.stdin.read()); code=payload["code"]; task=payload["task"]
def _piv_audit(event,args):
    try:
        if event=="open" and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event in {"os.listdir","os.scandir","os.chdir","os.remove","os.rename","os.rmdir","os.mkdir"} and args:
            p=args[0]
            if isinstance(p,(str,bytes,os.PathLike)) and os.path.abspath(os.fsdecode(p)).startswith("/kaggle"):
                raise PermissionError("PIV verifier blocks host /kaggle filesystem access")
        if event=="subprocess.Popen" or event=="os.system" or event.startswith("socket."):
            raise PermissionError("PIV verifier blocks process/network access")
    except PermissionError:
        raise
    except Exception:
        pass
sys.addaudithook(_piv_audit)
ns={"__name__":"__main__"}
def clean(g):
    try:
        import numpy as np
        if isinstance(g,np.ndarray): g=g.tolist()
    except Exception:
        pass
    # Source clean_grid accepts iterable rows. Normalize common iterable forms before LOCAL shape/domain checks.
    try:
        rows=list(g)
    except Exception:
        raise ValueError("invalid_grid_rows")
    if not (1<=len(rows)<=30):
        raise ValueError("invalid_grid_rows")
    out=[]
    for r in rows:
        try:
            if hasattr(r,"tolist"):
                r=r.tolist()
            rr0=list(r)
        except Exception:
            raise ValueError("invalid_grid_rows")
        out.append(rr0)
    widths=[len(r) for r in out]
    if not widths or min(widths)<1 or max(widths)>30 or len(set(widths))!=1:
        raise ValueError("invalid_grid_shape")
    clean_out=[]
    for r in out:
        rr=[]
        for x in r:
            if isinstance(x,bool) or not isinstance(x,numbers.Integral): raise ValueError("non_integer_cell")
            x=int(x)
            if x<0 or x>9: raise ValueError("cell_out_of_domain")
            rr.append(x)
        clean_out.append(rr)
    return clean_out

RESULT_SENTINEL="PIV_RESULT:"
result={"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],
        "test_execution_ok":False,"any_test_output_ok":False,"test_errors":[],
        "train_error":None,"test_error":None}
def emit_result():
    # Leading newline keeps the machine-readable result separate even if model code used print(..., end="").
    sys.__stdout__.write("\n"+RESULT_SENTINEL+json.dumps(result)+"\n")
    sys.__stdout__.flush()
try:
    exec(code,ns,ns); fn=ns.get("solution")
    if not callable(fn): raise ValueError("solution_not_callable")
except BaseException:
    result["train_error"]=traceback.format_exc(limit=10)
    emit_result(); raise SystemExit(0)

# SOURCE-FAITHFUL ordering: establish the train gate first.
for i,ex in enumerate(task["train"]):
    try:
        pred=clean(fn(copy.deepcopy(ex["input"])))
        exact=(pred==ex["output"])
        result["train"].append({"index":i,"exact":exact,"pred":pred if not exact else None,"error":None})
    except BaseException:
        result["train"].append({"index":i,"exact":False,"pred":None,"error":traceback.format_exc(limit=8)})
result["all_train_exact"]=bool(result["train"]) and all(x["exact"] for x in result["train"])

# Test execution is a separate per-output state, matching source produce_output_grid.py.
# One failing test input must not erase successful outputs for other test inputs.
if result["all_train_exact"]:
    tests=[]; test_errors=[]
    for j,ex in enumerate(task["test"]):
        try:
            tests.append(clean(fn(copy.deepcopy(ex["input"]))))
            test_errors.append(None)
        except BaseException:
            tests.append(None)
            test_errors.append(traceback.format_exc(limit=10))
    result["test_outputs"]=tests
    result["test_errors"]=test_errors
    result["test_execution_ok"]=all(x is not None for x in tests)
    result["any_test_output_ok"]=any(x is not None for x in tests)
    result["ok"]=result["any_test_output_ok"]
    if not result["test_execution_ok"]:
        result["test_error"]="\n".join(e for e in test_errors if e)

emit_result()
"""

def task_without_test_solutions(task):
    return {"train":copy.deepcopy(task["train"]),"test":[{"input":copy.deepcopy(x["input"])} for x in task["test"]]}

def fresh_verify_solution(code,task,timeout_sec=VERIFIER_TIMEOUT_SEC):
    ok,reason=guard_python_code(code,final=True)
    if not ok: return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":reason,"verifier_status":"guard_rejected"}
    payload={"code":code,"task":task_without_test_solutions(task)}
    try:
        cp=subprocess.run([sys.executable,"-u","-c",VERIFIER_SCRIPT],input=json.dumps(payload),text=True,capture_output=True,
                          timeout=float(timeout_sec),cwd=tempfile.mkdtemp(prefix="piv_verify_"))
        if cp.returncode!=0: return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":cp.stderr[-4000:],"verifier_status":"process_error"}
        sentinel="PIV_RESULT:"
        payload_lines=[ln[len(sentinel):] for ln in cp.stdout.splitlines() if ln.startswith(sentinel)]
        if not payload_lines:
            return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],
                    "error":"missing_verifier_result_sentinel","verifier_status":"protocol_error",
                    "stdout_tail":cp.stdout[-2000:]}
        r=json.loads(payload_lines[-1])
        if r.get("all_train_exact") and r.get("test_execution_ok"):
            r["verifier_status"]="train_exact_test_ok"
        elif r.get("all_train_exact") and r.get("any_test_output_ok"):
            r["verifier_status"]="train_exact_test_partial"
        elif r.get("all_train_exact"):
            r["verifier_status"]="train_exact_test_failed"
        else:
            r["verifier_status"]="train_rejected"
        return r
    except subprocess.TimeoutExpired:
        return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":"fresh_verifier_timeout","verifier_status":"timeout"}
    except Exception as e:
        return {"ok":False,"all_train_exact":False,"train":[],"test_outputs":[],"error":f"{type(e).__name__}: {e}","verifier_status":"error"}

# ---------------------------- PRELOAD SELF-TESTS ----------------------------
self_tests={}
t=parse_agent_turn("reasoning\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>")
assert t["kind"]=="tool" and len(t["tool_calls"])==1 and t["tool_name"]=="python" and "puzzle.keys" in t["tool_args"]["code"]
t_multi=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nx=1\n</parameter>\n</function>\n</tool_call>\n<tool_call>\n<function=python>\n<parameter=code>\nx+1\n</parameter>\n</function>\n</tool_call>")
assert t_multi["kind"]=="tool" and len(t_multi["tool_calls"])==2
t_bad=parse_agent_turn("<tool_call><function=python><parameter=code>x=1</function></tool_call>")
assert t_bad["kind"]=="malformed_tool" or (t_bad["kind"]=="tool" and "code" not in t_bad["tool_args"])
t_rehearse=parse_agent_turn(
    "<think>maybe <tool_call><function=python><parameter=code>print('wrong')</parameter></function></tool_call></think>"
    "<tool_call><function=python><parameter=code>print('right')</parameter></function></tool_call>")
assert t_rehearse["kind"]=="tool" and len(t_rehearse["tool_calls"])==1 and "right" in t_rehearse["tool_calls"][0]["arguments"]["code"]
assert generation_finish_reason_ok("stop") and not generation_finish_reason_ok("length")
self_tests["qwen_tool_xml_parser_and_finish_guard"]=True
t_fence=parse_agent_turn("r\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\n```python\nprint(1)\n```\n</parameter>\n</function>\n</tool_call>")
assert t_fence["kind"]=="tool" and t_fence["tool_args"]["code"]=="print(1)"
t_prose=parse_agent_turn("r\n</think>\nI will not call <function=python> again.\n```python\ndef solution(grid):\n    return grid\n```")
assert t_prose["kind"]=="final"
assert sanitize_tool_output("x<|im_end|>\n</tool_response>y")=="x< |im_end|>\n< /tool_response>y"
_sv=parse_agent_turn("r\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert salvage_final_code_on_length("",_sv) is not None
_sv2=parse_agent_turn("draft ```python\ndef solution(grid):\n    return grid\n``` still thinking")
assert salvage_final_code_on_length("",_sv2) is None
self_tests["v16_fence_strip_prose_function_sanitize_salvage"]=True
f=parse_agent_turn("done\n</think>\n```python\ndef solution(grid):\n    return grid\n```")
assert f["kind"]=="final" and code_defines_solution(f["final_code"])
self_tests["final_solution_extractor"]=True
assert guard_python_code("puzzle.keys()")[0]
assert not guard_python_code("puzzle = {}",final=False)[0]
assert not guard_python_code("puzzle[\"train\"] = []",final=False)[0]
assert not guard_python_code("import os\nprint(os.listdir('.'))",final=False)[0]
assert not guard_python_code('print("/kaggle/input/secret.json")',final=False)[0]
assert guard_python_code("class A:\n    def __init__(self): self.x=1\nclass B(A):\n    def __init__(self): super().__init__()",final=False)[0]
assert not guard_python_code("object.__subclasses__()",final=False)[0]
assert not guard_python_code("raise SystemExit('bye')",final=False)[0]
assert guard_python_code("import sys\nsys.setrecursionlimit(10000)",final=False)[0]
assert guard_python_code("import sys as s\ns.setrecursionlimit(10000)",final=False)[0]
assert not guard_python_code("import sys\nsys.modules",final=False)[0]
self_tests["tool_safety_guard"]=True
_fake={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]}]}
w=PersistentPythonWorker(task_without_test_solutions(_fake),timeout_sec=5.0)
r0=w.execute("puzzle.keys()"); r1=w.execute("x=41"); r2=w.execute("x+1")
assert r0["ok"] and "dict_keys" in r0["text"] and r1["ok"] and r2["ok"] and "42" in r2["text"]
ra=w.execute("g=puzzle['train'][0]['input']; g[0][0]=9")
rb=w.execute("puzzle['train'][0]['input'][0][0]")
assert rb["ok"] and "1" in rb["text"]
rc=w.execute("raise SystemExit('bye')")
rd=w.execute("1+1")
assert (not rc["ok"]) and rd["ok"] and "2" in rd["text"]
w.close()
self_tests["persistent_repl_state_and_expression_display"]=True
v=fresh_verify_solution("def solution(grid):\n    return [row[:] for row in grid]",_fake,timeout_sec=5)
assert v["all_train_exact"] and v["test_outputs"]==[[[2]]]
v_noise=fresh_verify_solution(
    "print('noise', end='')\ndef solution(grid):\n    return [row[:] for row in grid]",
    _fake,timeout_sec=5)
assert v_noise["all_train_exact"] and v_noise["test_outputs"]==[[[2]]]
self_tests["verifier_result_sentinel"]=True
self_tests["fresh_exact_verifier"]=True
v2=fresh_verify_solution("def solution(grid):\n    return [[9]]",_fake,timeout_sec=5)
assert not v2["all_train_exact"] and v2["test_outputs"]==[]
self_tests["train_fail_blocks_test_execution"]=True
v3=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('test-only failure')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake,timeout_sec=5)
assert v3["all_train_exact"] and not v3["test_execution_ok"] and v3["test_outputs"]==[None]
self_tests["train_exact_survives_test_execution_failure"]=True
_fake_multi={"train":[{"input":[[1]],"output":[[1]]}],"test":[{"input":[[2]]},{"input":[[3]]}]}
v4=fresh_verify_solution(
    "def solution(grid):\n    if grid == [[2]]: raise RuntimeError('one test fails')\n    return tuple(tuple(x for x in row) for row in grid)",
    _fake_multi,timeout_sec=5)
assert v4["all_train_exact"] and v4["test_outputs"][0] is None and v4["test_outputs"][1]==[[3]]
self_tests["partial_multiquery_test_outputs_preserved"]=True
with open(WORK_ROOT/"preload_self_tests.json","w") as f: json.dump(self_tests,f,indent=2)
print("PIV preload self-tests PASS:",self_tests)


PIV preload self-tests PASS: {'qwen_tool_xml_parser_and_finish_guard': True, 'v16_fence_strip_prose_function_sanitize_salvage': True, 'final_solution_extractor': True, 'tool_safety_guard': True, 'persistent_repl_state_and_expression_display': True, 'verifier_result_sentinel': True, 'fresh_exact_verifier': True, 'train_fail_blocks_test_execution': True, 'train_exact_survives_test_execution_failure': True, 'partial_multiquery_test_outputs_preserved': True}


In [7]:
# =====================================================================================
# CELL 7 — CPU-SAFE PREFLIGHT OR REAL vLLM/TP=4 MODEL LOAD (Q1-FROZEN ENGINE)
# Dependency versions were already pinned in Cell 1 for PREFLIGHT/production parity.
# =====================================================================================
gpu_info = []
notebook_start = NOTEBOOK_WALL_START
model_load_sec = 0.0
warmup_sec = 0.0
arc_llm = None

RUNTIME_VLLM_VERSION = pkg_version("vllm")
RUNTIME_TORCH_VERSION = pkg_version("torch")
RUNTIME_TRANSFORMERS_VERSION = pkg_version("transformers")
RUNTIME_TOKENIZERS_VERSION = pkg_version("tokenizers")
RUNTIME_HF_HUB_VERSION = pkg_version("huggingface-hub")
RUNTIME_CUDA_VERSION = None

deadline = notebook_start + TOTAL_RUNTIME_SEC - SAFETY_RESERVE_SEC

# Fingerprint the mounted checkpoint without hashing the huge shard bytes.
with open(index_path, "r", encoding="utf-8") as f:
    _model_index = json.load(f)
_model_shards = sorted(set((_model_index.get("weight_map") or {}).values()))
_model_shard_layout = []
for _name in _model_shards:
    _p = MODEL_PATH / _name
    _model_shard_layout.append({
        "name": _name,
        "bytes": int(_p.stat().st_size) if _p.exists() else None,
    })

MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {
    "model_path": str(MODEL_PATH),
    "config_sha256": small_file_sha256(config_path),
    "safetensors_index_sha256": small_file_sha256(index_path),
    "shard_layout": _model_shard_layout,
    "chat_template_sha256": template_sha,
    "tokenizer_class": tokenizer.__class__.__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
MODEL_ARTIFACT_FINGERPRINT_SHA256 = stable_json_sha256(MODEL_ARTIFACT_FINGERPRINT_PAYLOAD)

ENGINE_CONFIG = {
    "model": str(MODEL_PATH),
    "tensor_parallel_size": 4,
    "distributed_executor_backend": "mp",
    "dtype": "auto",
    "load_format": "safetensors",
    "trust_remote_code": False,
    "language_model_only": True,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "enable_prefix_caching": bool(ENABLE_PREFIX_CACHING),
    "seed": SEED,
}
ENGINE_CONFIG_SHA256 = stable_json_sha256(ENGINE_CONFIG)

if not USE_MODEL:
    # True no-allocation path: package parity is real, but do not import vLLM/CUDA runtime.
    SamplingParams = None
    print("=" * 116)
    print("PREFLIGHT CPU-SAFE / DEPENDENCY-PARITY MODE")
    print("=" * 116)
    print("vLLM package metadata :", RUNTIME_VLLM_VERSION)
    print("Transformers          :", RUNTIME_TRANSFORMERS_VERSION)
    print("Tokenizers            :", RUNTIME_TOKENIZERS_VERSION)
    print("HF Hub                :", RUNTIME_HF_HUB_VERSION)
    print("Skipping vLLM import, CUDA checks, SamplingParams API check and 27B allocation.")
    print("GPU/runtime-specific paths remain covered by SMOKE_GPU.")
else:
    import torch
    import vllm
    from vllm import LLM, SamplingParams

    RUNTIME_VLLM_VERSION = vllm.__version__
    RUNTIME_TORCH_VERSION = torch.__version__
    RUNTIME_CUDA_VERSION = torch.version.cuda

    print("=" * 116)
    print("REAL vLLM / CUDA PREFLIGHT")
    print("=" * 116)
    print("PyTorch :", RUNTIME_TORCH_VERSION)
    print("CUDA    :", RUNTIME_CUDA_VERSION)
    print("vLLM    :", RUNTIME_VLLM_VERSION)
    print("Transformers:", pkg_version("transformers"))

    if RUNTIME_VLLM_VERSION != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        raise RuntimeError("vLLM runtime drift after import.")
    if pkg_version("transformers") != EXPECTED_RUNTIME_PACKAGES["transformers"]:
        raise RuntimeError("Transformers runtime drift after vLLM setup.")

    # Fail fast on the exact SamplingParams fields used later, BEFORE allocating 27B.
    _sampling_api_probe = SamplingParams(
        temperature=1.0,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
        max_tokens=16,
        ignore_eos=False,
        seed=SEED,
        skip_special_tokens=False,
        spaces_between_special_tokens=False,
    )
    print("SamplingParams API self-test: PASS")

    print("\nLOADING QWEN3.8-27B-FP8 ON 4×L4")
    print("GPUs visible:", torch.cuda.device_count())

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable. Select Kaggle 4×L4 accelerator.")
    if torch.cuda.device_count() != 4:
        raise RuntimeError(f"Expected 4 GPUs; PyTorch sees {torch.cuda.device_count()}.")

    for i in range(4):
        cc = torch.cuda.get_device_capability(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        name = torch.cuda.get_device_name(i)
        gpu_info.append({"gpu": i, "name": name, "memory_gib": mem, "cc": f"{cc[0]}.{cc[1]}"})
        print(f"GPU {i}: {name} | {mem:.2f} GiB | CC {cc[0]}.{cc[1]}")

    load_start = time.perf_counter()
    arc_llm = LLM(**ENGINE_CONFIG)
    model_load_sec = time.perf_counter() - load_start
    print(f"MODEL LOAD COMPLETE: {model_load_sec:.1f} sec")

    # Real generation smoke request catches engine/scheduler/model problems immediately.
    _active_prompts = [r for r in prompt_records if r["task_id"] in ACTIVE_TASK_SET]
    _med = float(np.median([r["prompt_tokens"] for r in _active_prompts]))
    probe_record = min(_active_prompts, key=lambda r: abs(r["prompt_tokens"] - _med))
    warm_params = SamplingParams(
        temperature=0.0,
        max_tokens=32,
        ignore_eos=False,
        skip_special_tokens=False,
    )
    t0 = time.perf_counter()
    warm = arc_llm.generate([probe_record["prompt"]], [warm_params], use_tqdm=False)
    warmup_sec = time.perf_counter() - t0
    assert len(warm) == 1 and len(warm[0].outputs) >= 1
    print(f"REAL MODEL WARMUP (per-request SamplingParams sequence): PASS ({warmup_sec:.2f} sec)")

RUNTIME_STACK = {
    "vllm": pkg_version("vllm"),
    "torch": pkg_version("torch"),
    "transformers": pkg_version("transformers"),
    "tokenizers": pkg_version("tokenizers"),
    "huggingface-hub": pkg_version("huggingface-hub"),
    "cuda": RUNTIME_CUDA_VERSION,
}
RUNTIME_STACK_SHA256 = stable_json_sha256(RUNTIME_STACK)

print("Model artifact fingerprint:", MODEL_ARTIFACT_FINGERPRINT_SHA256)
print("Engine config fingerprint :", ENGINE_CONFIG_SHA256)
print("Runtime stack fingerprint :", RUNTIME_STACK_SHA256)
print("Safety reserve:", SAFETY_RESERVE_SEC / 60, "minutes")


PREFLIGHT CPU-SAFE / DEPENDENCY-PARITY MODE
vLLM package metadata : 0.19.0
Transformers          : 4.57.6
Tokenizers            : 0.22.2
HF Hub                : 0.36.2
Skipping vLLM import, CUDA checks, SamplingParams API check and 27B allocation.
GPU/runtime-specific paths remain covered by SMOKE_GPU.
Model artifact fingerprint: 1e0ec7cd18e73628ca3cb57482bad96788ccce4b3605b85a342e231d23d6f0e5
Engine config fingerprint : b1d41611a8c1f60b977fa205b112b5ee7663a318a97f2d090c21e78cadf4c866
Runtime stack fingerprint : 868ed7c0156e44dde22adbe151e74da9e898a504ac417bac9a8bbc9fd9c8db14
Safety reserve: 60.0 minutes


In [8]:
# =====================================================================================
# CELL 8 — PIV-Q27-P0 BOUNDED STATEFUL AGENT LOOP + CONTRACT-SAFE CHECKPOINTS
# =====================================================================================
RAW_GENERATIONS_PATH=WORK_ROOT/"raw_generations.jsonl"
TRACE_DIR=WORK_ROOT/"traces"; TRACE_DIR.mkdir(exist_ok=True)

PIV_SAMPLING={
    "temperature":1.0,"top_p":0.95,"top_k":20,"min_p":0.0,
    "presence_penalty":0.0,"repetition_penalty":1.0,"seed":SEED+3000,
    "per_turn_max_tokens":PER_TURN_MAX_TOKENS,
}
QWEN_OFFICIAL_SAMPLING_REFERENCE={"thinking":{k:PIV_SAMPLING[k] for k in ["temperature","top_p","top_k","min_p","presence_penalty","repetition_penalty"]}}
SAMPLING_CONFIG_SHA256=stable_json_sha256(PIV_SAMPLING)
AGENT_BUDGET={
    "agent_batch_size":AGENT_BATCH_SIZE,"max_tool_calls":MAX_TOOL_CALLS,"max_model_turns":MAX_MODEL_TURNS,
    "max_retries":MAX_RETRIES,
    "per_turn_max_tokens":PER_TURN_MAX_TOKENS,"total_generated_token_cap":TOTAL_GENERATED_TOKEN_CAP,
    "tool_timeout_sec":TOOL_TIMEOUT_SEC,"verifier_timeout_sec":VERIFIER_TIMEOUT_SEC,
    "max_tool_output_chars":MAX_TOOL_OUTPUT_CHARS,"history_policy":"preserve_thinking_and_tool_history",
}
AGENT_BUDGET_SHA256=stable_json_sha256(AGENT_BUDGET)

ENGINE_RUNTIME_CONTRACT={
    "model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
    "runtime_stack_sha256":RUNTIME_STACK_SHA256,
    "engine_config_sha256":ENGINE_CONFIG_SHA256,
    "chat_template_sha256":template_sha,
    "tool_schema_sha256":TOOL_SCHEMA_SHA256,
}

def generation_contract_payload(task_id):
    pr=PROMPT_RECORD_MAP[task_id]
    return {
        "generation_contract_version":GENERATION_CONTRACT_VERSION,"parser_contract_version":PARSER_CONTRACT_VERSION,
        "prompt_contract_version":PROMPT_CONTRACT_VERSION,"experiment_id":EXPERIMENT_ID,"run_profile":RUN_PROFILE,
        "task_set":TASK_SET,"task_id":task_id,"n_train":pr["n_train"],"n_test":pr["n_test"],
        "initial_prompt_sha256":pr["prompt_sha256"],"piv_source_commit":PIV_SOURCE_COMMIT,
        "sampling_config_sha256":SAMPLING_CONFIG_SHA256,"agent_budget_sha256":AGENT_BUDGET_SHA256,
        **ENGINE_RUNTIME_CONTRACT,
    }
def current_contract_sha(task_id): return stable_json_sha256(generation_contract_payload(task_id))

def append_record(rec):
    with open(RAW_GENERATIONS_PATH,"a",encoding="utf-8") as f:
        f.write(json.dumps(rec,ensure_ascii=False,default=str)+"\n")

def read_existing_records():
    if not RAW_GENERATIONS_PATH.exists(): return []
    out=[]
    for i,line in enumerate(RAW_GENERATIONS_PATH.read_text(encoding="utf-8").splitlines()):
        if not line.strip(): continue
        try:
            r=json.loads(line); r["_checkpoint_order"]=i; out.append(r)
        except Exception: pass
    return out

def record_compatible(r):
    tid=r.get("task_id")
    if tid not in ACTIVE_TASK_SET: return False,"foreign_task"
    if r.get("generation_contract_sha256") != current_contract_sha(tid): return False,"contract_mismatch"
    return True,"ok"

def audit_checkpoint_records(records):
    compatible=[]; stale=[]; foreign=[]
    for r in records:
        ok,reason=record_compatible(r)
        if ok: compatible.append(r)
        elif reason=="foreign_task": foreign.append({"task_id":r.get("task_id"),"record_id":r.get("record_id")})
        else: stale.append({"task_id":r.get("task_id"),"record_id":r.get("record_id"),"reason":reason})
    return compatible,{"raw_rows":len(records),"compatible_rows":len(compatible),"stale_rows":len(stale),"foreign_rows":len(foreign),
                       "stale_preview":stale[:20],"foreign_preview":foreign[:20]}

def terminal_completed_task_ids(records):
    out=set()
    for r in records:
        ok,_=record_compatible(r)
        if ok and r.get("status")=="complete": out.add(r["task_id"])
    return out

def save_event(task_id,event):
    p=TRACE_DIR/f"{task_id}.jsonl"
    with open(p,"a",encoding="utf-8") as f: f.write(json.dumps(event,ensure_ascii=False,default=str)+"\n")

def assistant_tool_message(parsed):
    calls=[]
    for tc in parsed.get("tool_calls",[]):
        calls.append({"type":"function","function":{"name":tc["name"],"arguments":tc["arguments"]}})
    return {
        "role":"assistant","content":parsed.get("content","") or "",
        "reasoning_content":parsed.get("reasoning_content","") or "",
        "tool_calls":calls,
    }

def complete_record(state, task, verifier, candidate_status, stop_reason, error=""):
    test_outputs=verifier.get("test_outputs",[]) if verifier else []
    code=state.get("final_code")
    rec={
        "run_id":RUN_ID,"record_id":str(uuid.uuid4()),"task_id":state["task_id"],"branch":"piv_p0","status":"complete",
        "candidate_status":candidate_status,"stop_reason":stop_reason,"error":error,
        "n_train":len(task["train"]),"n_test":len(task["test"]),"seed":PIV_SAMPLING["seed"],
        "sampling_seeds":state.get("sampling_seeds",[]),"source_retries":state.get("source_retries",0),
        "step_retries_total":state.get("step_retries_total",0),"retry_reasons":state.get("retry_reasons",[]),
        "budget_denials":state.get("budget_denials",0),
        "guard_rejections":state.get("guard_rejections",0),"salvaged_final_on_length":state.get("salvaged_final_on_length",0),
        "model_turns":state["model_turns"],"tool_calls":state["tool_calls"],"tool_errors":state["tool_errors"],
        "tool_timeouts":state["tool_timeouts"],"worker_restarts":state["worker_restarts"],
        "generated_tokens":state["generated_tokens"],"max_rendered_prompt_tokens":state["max_prompt_tokens"],
        "finish_reasons":state["finish_reasons"],"final_code":code,"final_code_sha256":hashlib.sha256((code or "").encode()).hexdigest() if code else None,
        "all_train_exact":bool(verifier and verifier.get("all_train_exact")),
        "test_execution_ok":bool(verifier and verifier.get("test_execution_ok")),
        "any_test_output_ok":bool(verifier and verifier.get("any_test_output_ok")),
        "test_errors":verifier.get("test_errors",[]) if verifier else [],
        "verifier_status":verifier.get("verifier_status") if verifier else None,
        "verifier_error":(verifier.get("train_error") or verifier.get("test_error") or verifier.get("error")) if verifier else None,
        "test_outputs":test_outputs,
        "initial_prompt_sha256":PROMPT_RECORD_MAP[state["task_id"]]["prompt_sha256"],
        "generation_contract_sha256":current_contract_sha(state["task_id"]),"sampling_config_sha256":SAMPLING_CONFIG_SHA256,
        "agent_budget_sha256":AGENT_BUDGET_SHA256,"piv_source_commit":PIV_SOURCE_COMMIT,**ENGINE_RUNTIME_CONTRACT,
    }
    return rec

def step_retry_available(state):
    return (
        state.get("step_retry_count",0) < MAX_RETRIES
        and state.get("generated_tokens",0) < TOTAL_GENERATED_TOKEN_CAP
        and state.get("model_turns",0) < MAX_MODEL_TURNS
    )

def register_step_retry(state, reason):
    """PiVOTools inner tool-loop retry: same messages + same persistent REPL state."""
    if not step_retry_available(state):
        return False
    state["step_retry_count"] = int(state.get("step_retry_count",0)) + 1
    state["step_retries_total"] = int(state.get("step_retries_total",0)) + 1
    state["sampling_retry_nonce"] = int(state.get("sampling_retry_nonce",0)) + 1
    state.setdefault("retry_reasons",[]).append(f"step:{reason}")
    return True

def run_retry_policy_preflight_test():
    st={"step_retry_count":0,"step_retries_total":0,"sampling_retry_nonce":0,
        "generated_tokens":100,"model_turns":1,"tool_calls":3,
        "messages":[{"role":"user","content":"sentinel-history"}],"retry_reasons":[]}
    before_messages=copy.deepcopy(st["messages"]); before_calls=st["tool_calls"]
    assert register_step_retry(st,"invalid_finish_reason:length")
    assert st["messages"]==before_messages and st["tool_calls"]==before_calls
    assert st["step_retry_count"]==1 and st["sampling_retry_nonce"]==1
    assert register_step_retry(st,"invalid_finish_reason:length")
    assert not register_step_retry(st,"invalid_finish_reason:length")
    return {"ok":True,"step_retries_total":st["step_retries_total"],
            "history_preserved":st["messages"]==before_messages,"tool_calls_preserved":st["tool_calls"]==before_calls}

def run_fake_scheduler_preflight_test():
    """Scripted length -> same-history retry -> Python tool -> final program -> fresh verifier."""
    fake={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]}],
          "test":[{"input":[[5,6],[7,8]]}]}
    st={"step_retry_count":0,"step_retries_total":0,"sampling_retry_nonce":0,
        "generated_tokens":200,"model_turns":1,"tool_calls":0,
        "messages":[{"role":"user","content":"fake"}],"retry_reasons":[]}
    history_before=copy.deepcopy(st["messages"])
    assert register_step_retry(st,"invalid_finish_reason:length")
    assert st["messages"]==history_before
    worker=PersistentPythonWorker(task_without_test_solutions(fake),timeout_sec=5.0)
    try:
        tool_text="inspect\n</think>\n<tool_call><function=python><parameter=code>puzzle.keys()</parameter></function></tool_call>"
        p=parse_agent_turn(tool_text); assert p["kind"]=="tool"
        st["step_retry_count"]=0
        rr=worker.execute(p["tool_calls"][0]["arguments"]["code"]); assert rr["ok"]
        st["tool_calls"]+=1
        final_text="done\n</think>\n```python\ndef solution(grid):\n    return [row[:] for row in grid]\n```"
        pf=parse_agent_turn(final_text); assert pf["kind"]=="final"
        vr=fresh_verify_solution(pf["final_code"],fake,timeout_sec=5)
        assert vr["all_train_exact"] and vr["test_execution_ok"]
    finally:
        worker.close()
    return {"ok":True,"tool_calls":st["tool_calls"],"verifier_status":vr["verifier_status"]}

def run_scripted_preflight_agent_loop_test():
    """CPU-only scripted tool->feedback->final-code path using the same parser/worker/verifier pieces."""
    fake={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]}],
          "test":[{"input":[[5,6],[7,8]]}]}
    worker=PersistentPythonWorker(task_without_test_solutions(fake),timeout_sec=5.0)
    try:
        raw1="inspect\n</think>\n<tool_call><function=python><parameter=code>puzzle.keys()</parameter></function></tool_call>"
        p1=parse_agent_turn(raw1)
        assert p1["kind"]=="tool" and len(p1["tool_calls"])==1
        rr=worker.execute(p1["tool_calls"][0]["arguments"]["code"])
        assert rr["ok"] and "dict_keys" in rr["text"]
        raw2="done\n</think>\n```python\ndef solution(grid):\n    return [row[:] for row in grid]\n```"
        p2=parse_agent_turn(raw2)
        assert p2["kind"]=="final"
        vr=fresh_verify_solution(p2["final_code"],fake,timeout_sec=5)
        assert vr["all_train_exact"] and vr["test_outputs"]==[[[5,6],[7,8]]]
        payload={"ok":True,"tool_feedback":rr["text"],"verifier_status":vr["verifier_status"]}
    finally:
        worker.close()
    with open(WORK_ROOT/"scripted_agent_loop_selftest.json","w") as f:
        json.dump(payload,f,indent=2)
    print("Scripted CPU agent-loop self-test PASS:",payload)

def run_fake_llm_scheduler_preflight_test():
    """v1.6: drive the REAL run_real_piv() scheduler on CPU with a scripted fake LLM and the real REPL worker,
    parser and fresh verifier. Globals are swapped for the duration so PREFLIGHT mock records stay untouched."""
    _g=globals()
    _saved={k:_g.get(k) for k in ["arc_llm","SamplingParams","ACTIVE_TASK_IDS","ACTIVE_TASK_SET","deadline",
                                  "RAW_GENERATIONS_PATH","TRACE_DIR","WAVE_METRICS_PATH","AGENT_LOOP_RUNTIME_PATH"]}
    fake_task={"train":[{"input":[[1,2],[3,4]],"output":[[1,2],[3,4]]},{"input":[[5,0],[0,5]],"output":[[5,0],[0,5]]}],
               "test":[{"input":[[7,7],[8,9]]}]}
    fake_ids=["fakeA_happy","fakeB_budget_death","fakeC_wrong_tool_name","fakeD_guard_rejected","fakeE_length_salvage"]
    TOOL_KEYS="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>"
    TOOL_LEN="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nprint(len(puzzle[\"train\"]))\n</parameter>\n</function>\n</tool_call>"
    TOOL_OS="inspect\n</think>\n<tool_call>\n<function=python>\n<parameter=code>\nimport os\nprint(os.getcwd())\n</parameter>\n</function>\n</tool_call>"
    WRONG="inspect\n</think>\n<tool_call>\n<function=python2>\n<parameter=code>\npuzzle.keys()\n</parameter>\n</function>\n</tool_call>"
    FINAL="done\n</think>\n```python\ndef solution(grid):\n    return [row[:] for row in grid]\n```"
    def script(tid,n_tool,cap):
        if tid=="fakeA_happy":   return (TOOL_KEYS,600,"stop") if n_tool==0 else ((TOOL_LEN,600,"stop") if n_tool==1 else (FINAL,400,"stop"))
        if tid=="fakeB_budget_death": return (TOOL_KEYS,600,"stop") if n_tool==0 else ("still reasoning about the rule ...",cap,"length")
        if tid=="fakeC_wrong_tool_name": return (WRONG,300,"stop")
        if tid=="fakeD_guard_rejected": return (TOOL_OS,300,"stop") if n_tool==0 else (FINAL,400,"stop")
        if tid=="fakeE_length_salvage": return (FINAL,500,"length")
        raise RuntimeError(tid)
    class _Out:
        def __init__(s,text,n,fr): s.text=text; s.token_ids=[0]*int(n); s.finish_reason=fr
    class _Req:
        def __init__(s,o): s.outputs=[o]
    class _FakeSP:
        def __init__(s,**kw): s.__dict__.update(kw)
    prefix_keys={}
    class _FakeLLM:
        def generate(s,prompts,params,use_tqdm=False):
            outs=[]
            for p,sp in zip(prompts,params):
                tid=next(t for t,pref in prefix_keys.items() if p.startswith(pref))
                text,n,fr=script(tid,p.count("<tool_response>"),int(sp.max_tokens))
                outs.append(_Req(_Out(text,min(int(n),int(sp.max_tokens)),fr)))
            return outs
    tmp=Path(tempfile.mkdtemp(prefix="piv_fake_sched_"))
    added_prompt=[]; added_task=[]
    try:
        for tid in fake_ids:
            eval_challenges[tid]=copy.deepcopy(fake_task); added_task.append(tid)
            # All fake lineages share one puzzle; tag the prompt so the fake LLM can route by rendered prefix.
            _user=build_agent_user_prompt(fake_task)+f"\n\n[PIV PREFLIGHT fake lineage: {tid}]"
            _msgs=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":_user}]
            _rendered,_m=render_agent_chat(_msgs)
            PROMPT_RECORD_MAP[tid]={"task_id":tid,"user_prompt":_user,"initial_messages":_msgs,"prompt":_rendered,
                                    "prompt_sha256":hashlib.sha256(_rendered.encode()).hexdigest(),
                                    "prompt_tokens":len(tokenizer.encode(_rendered,add_special_tokens=False)),
                                    "n_train":2,"n_test":1,"render_method":_m}
            added_prompt.append(tid)
            prefix_keys[tid]=_rendered[:_rendered.rfind("<|im_start|>assistant")]
        _g["arc_llm"]=_FakeLLM(); _g["SamplingParams"]=_FakeSP
        _g["ACTIVE_TASK_IDS"]=list(fake_ids); _g["ACTIVE_TASK_SET"]=set(fake_ids)
        _g["RAW_GENERATIONS_PATH"]=tmp/"raw.jsonl"; _g["TRACE_DIR"]=tmp/"traces"; (tmp/"traces").mkdir()
        _g["WAVE_METRICS_PATH"]=tmp/"wave.jsonl"; _g["AGENT_LOOP_RUNTIME_PATH"]=tmp/"rt.json"
        _g["deadline"]=time.perf_counter()+3600.0
        run_real_piv()
        recs={json.loads(l)["task_id"]:json.loads(l) for l in open(tmp/"raw.jsonl",encoding="utf-8")}
        waves=[json.loads(l) for l in open(tmp/"wave.jsonl",encoding="utf-8")]
        A,B,C,D,E=(recs[t] for t in fake_ids)
        assert A["candidate_status"]=="verified" and A["tool_calls"]==2 and A["all_train_exact"] and A["test_outputs"]==[[[7,7],[8,9]]], A
        assert B["stop_reason"]=="invalid_finish_reason" and B["generated_tokens"]==TOTAL_GENERATED_TOKEN_CAP and B["tool_calls"]==1, B
        assert C["tool_calls"]==0 and C["step_retries_total"]>=2 and C["tool_errors"]>=1 and C["stop_reason"]=="agent_budget_exhausted", C
        # retry counter regression: never more than MAX_RETRIES consecutive step retries before a feedback turn
        _cev=[json.loads(l) for l in open(tmp/"traces"/"fakeC_wrong_tool_name.jsonl",encoding="utf-8")]
        _run=0; _max_run=0
        for ev in _cev:
            if ev.get("step_retry"): _run+=1; _max_run=max(_max_run,_run)
            else: _run=0
        assert _max_run<=MAX_RETRIES, ("consecutive step retries exceeded MAX_RETRIES",_max_run)
        assert D["candidate_status"]=="verified" and D["tool_calls"]==0 and D["guard_rejections"]==1, D
        assert E["candidate_status"]=="verified" and E["salvaged_final_on_length"]==1 and E["finish_reasons"]==["length"], E
        assert len(waves)>=3 and all(w.get("sec_per_output_token_longest") is not None for w in waves), waves[:1]
        # deadline pre-check: a deadline closer than one projected wave must fail cleanly and stop.
        (tmp/"raw.jsonl").unlink()
        _g["deadline"]=time.perf_counter()+120.0
        run_real_piv()
        recs2=[json.loads(l) for l in open(tmp/"raw.jsonl",encoding="utf-8")]
        assert len(recs2)==len(fake_ids) and all(r["status"]=="failed" and r["stop_reason"]=="global_deadline_projected" for r in recs2), recs2[:1]
        result={"ok":True,"happy":A["candidate_status"],"budget_death":B["stop_reason"],
                "wrong_tool_max_consecutive_step_retries":_max_run,"wrong_tool_step_retries_total":C["step_retries_total"],
                "guard_not_charged":{"tool_calls":D["tool_calls"],"guard_rejections":D["guard_rejections"]},
                "length_salvage":E["candidate_status"],"deadline_precheck":"all_failed_global_deadline_projected","waves":len(waves)}
    finally:
        for tid in added_task: eval_challenges.pop(tid,None)
        for tid in added_prompt: PROMPT_RECORD_MAP.pop(tid,None)
        for k,v in _saved.items(): _g[k]=v
    with open(WORK_ROOT/"fake_llm_scheduler_selftest.json","w") as f: json.dump(result,f,indent=2)
    print("Fake-LLM REAL-scheduler PREFLIGHT self-test PASS:",result)

def run_multicall_budget_accounting_selftest():
    max_calls=1
    executed=0
    outcomes=[]
    for guard_rejected in (True,False):
        if executed>=max_calls:
            outcomes.append("denied")
            continue
        if guard_rejected:
            outcomes.append("guard_rejected")
        else:
            executed+=1
            outcomes.append("executed")
    assert outcomes==["guard_rejected","executed"] and executed==1
    return {"ok":True,"outcomes":outcomes,"executed_calls":executed}

def run_mock_preflight():
    retry_test=run_retry_policy_preflight_test()
    fake_scheduler_test=run_fake_scheduler_preflight_test()
    multicall_budget_test=run_multicall_budget_accounting_selftest()
    with open(WORK_ROOT/"retry_policy_selftest.json","w") as f:
        json.dump({"retry_policy":retry_test,"fake_scheduler":fake_scheduler_test,
                   "multicall_budget":multicall_budget_test},f,indent=2)
    print("Retry/scheduler PREFLIGHT self-tests PASS:",retry_test,fake_scheduler_test,multicall_budget_test)
    run_scripted_preflight_agent_loop_test()
    run_fake_llm_scheduler_preflight_test()
    # Exercise downstream checkpoint/scoring paths without pretending to measure Qwen accuracy.
    for tid in ACTIVE_TASK_IDS:
        state={"task_id":tid,"model_turns":0,"tool_calls":0,"tool_errors":0,"tool_timeouts":0,"worker_restarts":0,
               "generated_tokens":0,"max_prompt_tokens":PROMPT_RECORD_MAP[tid]["prompt_tokens"],
               "finish_reasons":["mock_preflight"],"final_code":None,"sampling_seeds":[],
               "source_retries":0,"step_retries_total":0,"retry_reasons":[],"budget_denials":0,
               "guard_rejections":0,"salvaged_final_on_length":0}
        rec=complete_record(state,eval_challenges[tid],None,"mock_rejected","mock_preflight")
        append_record(rec)
    print("PREFLIGHT mock records written:",len(ACTIVE_TASK_IDS))

WAVE_METRICS_PATH = WORK_ROOT / "wave_metrics.jsonl"
AGENT_LOOP_RUNTIME_PATH = WORK_ROOT / "agent_loop_runtime.json"

def append_wave_metric(row):
    with open(WAVE_METRICS_PATH,"a",encoding="utf-8") as _f:
        _f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n")

def run_real_piv():
    loop_t0=time.perf_counter()
    pre_loop_elapsed_sec=loop_t0-notebook_start
    global_wave_no=0
    wave_spt_samples=[]      # v1.6: measured seconds per output token of each wave's longest sequence
    stop_all=False
    existing=read_existing_records(); completed=terminal_completed_task_ids(existing)
    pending=[t for t in ACTIVE_TASK_IDS if t not in completed]
    print("Resume: completed",len(completed),"pending",len(pending))

    for batch_no,start_idx in enumerate(range(0,len(pending),AGENT_BATCH_SIZE),1):
        tids=pending[start_idx:start_idx+AGENT_BATCH_SIZE]
        batch_wave_no=0
        if stop_all:
            print("Projected-deadline stop already triggered; not starting batch",batch_no); break
        if time.perf_counter()>=deadline:
            print("Safety deadline reached before batch",batch_no); break
        workers={}; states={}

        def start_fresh_attempt(tid, st, reason=None):
            # PiVOTools AbstractTurn retries invalid/incomplete agentic calls from the initial prompt.
            # LOCAL fairness: retries share the same total token/tool/turn budgets; they do not create extra candidates.
            if reason is not None:
                if st["source_retries"]>=MAX_RETRIES:
                    return False
                if st["generated_tokens"]>=TOTAL_GENERATED_TOKEN_CAP or st["model_turns"]>=MAX_MODEL_TURNS:
                    return False
                st["source_retries"]+=1
                st["sampling_retry_nonce"]+=1
                st["retry_reasons"].append(str(reason))
            old=workers.get(tid)
            if old is not None:
                st["worker_restart_offset"] += int(getattr(old,"restart_count",0))
                st["worker_restarts"] = st["worker_restart_offset"]
                try: old.close()
                except Exception: pass
            workers[tid]=PersistentPythonWorker(task_without_test_solutions(eval_challenges[tid]),TOOL_TIMEOUT_SEC)
            st["messages"]=copy.deepcopy(PROMPT_RECORD_MAP[tid]["initial_messages"])
            st["attempt_turns"]=0
            st["step_retry_count"]=0
            st["budget_nudged"]=False
            st["final_code"]=None
            return True

        try:
            for tid in tids:
                states[tid]={
                    "task_id":tid,"messages":[],"model_turns":0,"attempt_turns":0,
                    "tool_calls":0,"tool_errors":0,"tool_timeouts":0,"worker_restarts":0,"worker_restart_offset":0,
                    "generated_tokens":0,"max_prompt_tokens":0,"finish_reasons":[],"done":False,"final_code":None,
                    "source_retries":0,"step_retry_count":0,"step_retries_total":0,"sampling_retry_nonce":0,
                    "sampling_seeds":[],"retry_reasons":[],"budget_nudged":False,"budget_denials":0,
                    "guard_rejections":0,"salvaged_final_on_length":0,
                }
                start_fresh_attempt(tid,states[tid])

            while True:
                active=[t for t in tids if not states[t]["done"]]
                if not active: break

                if time.perf_counter()>=deadline:
                    for tid in active:
                        st=states[tid]; st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"failed","global_deadline","safety reserve reached")
                        rec["status"]="failed"; append_record(rec)
                    break

                render=[]; render_tids=[]; max_allowed=[]; request_seeds=[]
                for tid in active:
                    st=states[tid]
                    if st["model_turns"]>=MAX_MODEL_TURNS or st["generated_tokens"]>=TOTAL_GENERATED_TOKEN_CAP:
                        st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"rejected","agent_budget_exhausted")
                        append_record(rec); continue

                    prompt,method=render_agent_chat(st["messages"])
                    pt=len(tokenizer.encode(prompt,add_special_tokens=False))
                    st["max_prompt_tokens"]=max(st["max_prompt_tokens"],pt)
                    safe=MAX_MODEL_LEN-pt-CONTEXT_SAFETY_TOKENS
                    remaining=TOTAL_GENERATED_TOKEN_CAP-st["generated_tokens"]
                    cap=min(PER_TURN_MAX_TOKENS,remaining,safe)
                    if cap<32:
                        st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"rejected","context_or_token_budget_exhausted")
                        append_record(rec); continue

                    req_seed=int(PIV_SAMPLING["seed"]+st["sampling_retry_nonce"])
                    render.append(prompt); render_tids.append(tid); max_allowed.append(int(cap)); request_seeds.append(req_seed)
                    st["sampling_seeds"].append(req_seed)

                if not render_tids: continue

                # v1.6: a wave runs for up to max(caps) tokens. Refuse to start one that would cross the deadline;
                # the affected tasks are recorded as failed (resumable), never silently lost to a Kaggle kill.
                if wave_spt_samples:
                    _measured_p90=float(np.percentile(wave_spt_samples,90))
                    _spt=max(float(DEFAULT_SEC_PER_OUTPUT_TOKEN),_measured_p90)
                else:
                    _spt=float(DEFAULT_SEC_PER_OUTPUT_TOKEN)
                _projected_wave_sec=float(max(max_allowed))*_spt*float(WAVE_TIME_SAFETY_FACTOR)+float(WAVE_OVERHEAD_SEC)
                if time.perf_counter()+_projected_wave_sec>=deadline:
                    for tid in render_tids:
                        st=states[tid]; st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"failed","global_deadline_projected",
                                            f"projected wave {_projected_wave_sec:.0f}s at {_spt:.3f}s/token would cross the safety deadline")
                        rec["status"]="failed"; append_record(rec)
                    stop_all=True
                    print(f"batch {batch_no}: projected wave {_projected_wave_sec/60:.1f} min would cross the deadline; stopping cleanly.")
                    break

                params=[]
                for _cap,_seed in zip(max_allowed,request_seeds):
                    params.append(SamplingParams(
                        temperature=PIV_SAMPLING["temperature"],top_p=PIV_SAMPLING["top_p"],top_k=PIV_SAMPLING["top_k"],
                        min_p=PIV_SAMPLING["min_p"],presence_penalty=PIV_SAMPLING["presence_penalty"],
                        repetition_penalty=PIV_SAMPLING["repetition_penalty"],max_tokens=int(_cap),ignore_eos=False,
                        seed=int(_seed),skip_special_tokens=False,spaces_between_special_tokens=False))

                batch_wave_no+=1; global_wave_no+=1
                t0=time.perf_counter(); outs=arc_llm.generate(render,params,use_tqdm=False); wave_sec=time.perf_counter()-t0
                _wave_output_tokens=[]; _wave_finish=[]
                for _req in outs:
                    if getattr(_req,"outputs",None):
                        _o=_req.outputs[0]
                        _wave_output_tokens.append(len(getattr(_o,"token_ids",[]) or []))
                        _wave_finish.append(str(getattr(_o,"finish_reason",None)))
                    else:
                        _wave_output_tokens.append(0); _wave_finish.append("no_output")
                append_wave_metric({
                    "run_id":RUN_ID,"batch_no":batch_no,"batch_wave_no":batch_wave_no,"global_wave_no":global_wave_no,
                    "task_ids":list(render_tids),"active_count":len(render_tids),"caps":list(max_allowed),
                    "request_seeds":list(request_seeds),"output_tokens":_wave_output_tokens,
                    "finish_reasons":_wave_finish,"sum_output_tokens":int(sum(_wave_output_tokens)),
                    "max_output_tokens":int(max(_wave_output_tokens) if _wave_output_tokens else 0),
                    "wave_sec":float(wave_sec),
                    "aggregate_output_tokens_per_sec":float(sum(_wave_output_tokens)/wave_sec) if wave_sec>0 else None,
                    "sec_per_output_token_longest":(float(wave_sec)/float(max(_wave_output_tokens))) if (_wave_output_tokens and max(_wave_output_tokens)>0) else None,
                })
                if _wave_output_tokens and max(_wave_output_tokens)>0 and wave_sec>0:
                    wave_spt_samples.append(float(wave_sec)/float(max(_wave_output_tokens)))

                for tid,req in zip(render_tids,outs):
                    st=states[tid]; st["model_turns"]+=1; st["attempt_turns"]+=1

                    if not getattr(req,"outputs",None):
                        event={"run_id":RUN_ID,"task_id":tid,"turn":st["model_turns"],"attempt_turn":st["attempt_turns"],
                               "parsed_kind":"no_output","wave_sec":wave_sec}
                        if register_step_retry(st,"no_output"):
                            event["step_retry"]=True; save_event(tid,event); continue
                        if start_fresh_attempt(tid,st,"no_output"):
                            event["source_retry"]=True; save_event(tid,event); continue
                        st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"failed","no_output","vLLM returned no output")
                        rec["status"]="failed"; append_record(rec); save_event(tid,event); continue

                    o=req.outputs[0]; text=o.text or ""; nt=len(getattr(o,"token_ids",[]) or [])
                    st["generated_tokens"]+=nt
                    fr=str(getattr(o,"finish_reason",None)); st["finish_reasons"].append(fr)
                    parsed=parse_agent_turn(text)
                    event={"run_id":RUN_ID,"task_id":tid,"turn":st["model_turns"],"attempt_turn":st["attempt_turns"],
                           "source_retry_index":st["source_retries"],"prompt_tokens":st["max_prompt_tokens"],
                           "generated_tokens":nt,"finish_reason":fr,"raw_text":text,"parsed_kind":parsed["kind"],
                           "wave_sec":wave_sec,"request_seed":st["sampling_seeds"][-1]}

                    # PiVOTools has TWO retry layers: first retry the current tool-loop step with
                    # unchanged history/REPL, then let AbstractTurn restart the complete agentic call.
                    _salvaged=salvage_final_code_on_length(text,parsed) if str(fr).lower()=="length" else None
                    if _salvaged is not None:
                        st["salvaged_final_on_length"]+=1; event["salvaged_final_on_length"]=True
                    if not generation_finish_reason_ok(fr) and _salvaged is None:
                        if register_step_retry(st,f"invalid_finish_reason:{fr}"):
                            event["step_retry"]=True; save_event(tid,event); continue
                        if start_fresh_attempt(tid,st,f"invalid_finish_reason:{fr}"):
                            event["source_retry"]=True; save_event(tid,event); continue
                        st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"rejected","invalid_finish_reason",
                                            f"finish_reason={fr}; retries exhausted or budget unavailable")
                        append_record(rec); save_event(tid,event); continue

                    if parsed["kind"]=="malformed_tool":
                        if register_step_retry(st,"malformed_tool_xml"):
                            event["step_retry"]=True; save_event(tid,event); continue
                        if start_fresh_attempt(tid,st,"malformed_tool_xml"):
                            event["source_retry"]=True; save_event(tid,event); continue
                        st["tool_errors"]+=1; st["done"]=True
                        rec=complete_record(st,eval_challenges[tid],None,"rejected","malformed_tool_xml")
                        append_record(rec); save_event(tid,event); continue

                    if parsed["kind"]=="tool":
                        calls=parsed.get("tool_calls",[])

                        # Source first retries malformed/unsupported calls at the SAME tool-loop step.
                        malformed=any(tc.get("name")!="python" or "code" not in tc.get("arguments",{}) for tc in calls)
                        if malformed:
                            if register_step_retry(st,"malformed_or_unsupported_structured_tool"):
                                event["step_retry"]=True; save_event(tid,event); continue
                            st["step_retry_count"]=0
                            st["messages"].append(assistant_tool_message(parsed))
                            for tc in calls:
                                st["messages"].append({"role":"tool","content":
                                    f"Tool call rejected: expected python(code=...). Got name={tc.get('name')!r}, keys={list(tc.get('arguments',{}))}."})
                            st["tool_errors"]+=1
                            event["tool_result"]="unsupported_or_invalid_structured_tool_call"
                            save_event(tid,event); continue

                        st["step_retry_count"]=0   # v1.6: reset only once a VALID structured tool turn is accepted
                        st["messages"].append(assistant_tool_message(parsed))
                        event_calls=[]
                        denial=("LOCAL P0 tool budget exhausted. No further Python execution is available. "
                                "Return the self-contained final `solution(grid)` Python code block now.")
                        for call_idx,tc in enumerate(calls):
                            if st["tool_calls"] >= MAX_TOOL_CALLS:
                                st["messages"].append({"role":"tool","content":denial})
                                st["budget_denials"]+=1
                                event_calls.append({"tool_code":tc.get("arguments",{}).get("code"),
                                                    "tool_ok":False,"tool_output_to_model":denial,
                                                    "budget_denied":True})
                                continue
                            code=tc["arguments"]["code"]
                            rr=workers[tid].execute(code)
                            if rr.get("guard_rejected"):
                                # v1.6: a locally rejected snippet never reached Python; do not charge the 6-call budget.
                                st["guard_rejections"]+=1
                            else:
                                st["tool_calls"]+=1
                            # worker.restart_count is cumulative within the current source-attempt worker.
                            current_worker_restarts=int(getattr(workers[tid],"restart_count",0))
                            st["worker_restarts"]=st["worker_restart_offset"]+current_worker_restarts
                            if not rr.get("ok"): st["tool_errors"]+=1
                            if rr.get("timeout"): st["tool_timeouts"]+=1
                            raw_tool=str(rr.get("text",""))[:MAX_TOOL_RAW_ARTIFACT_CHARS]
                            model_tool=sanitize_tool_output(compact_tool_output(raw_tool))
                            if st["tool_calls"]>=MAX_TOOL_CALLS:
                                model_tool += ("\n\n[LOCAL P0 BUDGET] This was the final permitted Python execution. "
                                               "Your next assistant response must be the final self-contained `solution(grid)` code block.")
                            st["messages"].append({"role":"tool","content":model_tool})
                            event_calls.append({"tool_code":code,"tool_ok":rr.get("ok"),
                                                "tool_raw_output":raw_tool,"tool_output_to_model":model_tool,
                                                "current_worker_restarts":current_worker_restarts})
                        event["tool_calls_executed"]=event_calls
                        _denied_this_turn=sum(1 for x in event_calls if x.get("budget_denied"))
                        if _denied_this_turn:
                            event["tool_result"]="partial_tool_budget_denial_finalization_nudge"
                            if st["budget_nudged"] and st["tool_calls"]>=MAX_TOOL_CALLS:
                                st["done"]=True
                                rec=complete_record(st,eval_challenges[tid],None,"rejected","tool_budget_exhausted_after_nudge")
                                append_record(rec)
                            else:
                                st["budget_nudged"]=True
                        save_event(tid,event); continue

                    if parsed["kind"]=="final":
                        st["step_retry_count"]=0
                        code=parsed["final_code"]; st["final_code"]=code
                        verifier=fresh_verify_solution(code,eval_challenges[tid],VERIFIER_TIMEOUT_SEC)
                        if verifier.get("all_train_exact") and verifier.get("test_execution_ok"):
                            cand="verified"
                        elif verifier.get("all_train_exact") and verifier.get("any_test_output_ok"):
                            cand="verified_partial_test"
                        elif verifier.get("all_train_exact"):
                            cand="test_execution_failed"
                        else:
                            cand="train_rejected"
                        st["done"]=True; event["verifier_status"]=verifier.get("verifier_status")
                        rec=complete_record(st,eval_challenges[tid],verifier,cand,"final_program")
                        append_record(rec); save_event(tid,event); continue

                    # No tool and no valid final code: source CodeSolutionTurn rejects and AbstractTurn retries.
                    if start_fresh_attempt(tid,st,"no_tool_call_or_final_solution"):
                        event["source_retry"]=True; save_event(tid,event); continue
                    st["done"]=True
                    rec=complete_record(st,eval_challenges[tid],None,"rejected","no_tool_call_or_final_solution")
                    append_record(rec); save_event(tid,event)

                print(f"batch {batch_no}: active={sum(not states[t]['done'] for t in tids)} | "
                      f"wave={wave_sec:.1f}s | caps={min(max_allowed)}..{max(max_allowed)}")
        finally:
            for tid,w in workers.items():
                try: w.close()
                except Exception: pass

    loop_elapsed_sec=time.perf_counter()-loop_t0
    with open(AGENT_LOOP_RUNTIME_PATH,"w") as _f:
        json.dump({
            "run_id":RUN_ID,
            "pre_loop_elapsed_sec":float(pre_loop_elapsed_sec),
            "loop_elapsed_sec":float(loop_elapsed_sec),
            "agent_batch_size":AGENT_BATCH_SIZE,
            "active_tasks":len(ACTIVE_TASK_IDS),
            "global_waves":int(global_wave_no),
        },_f,indent=2)
    print("Agent loop runtime:",json.dumps({
        "pre_loop_elapsed_sec":round(pre_loop_elapsed_sec,2),
        "loop_elapsed_sec":round(loop_elapsed_sec,2),
        "global_waves":global_wave_no,
    }))

if not USE_MODEL: run_mock_preflight()
else: run_real_piv()
print("Cell 8 complete. Raw checkpoint:",RAW_GENERATIONS_PATH)


Retry/scheduler PREFLIGHT self-tests PASS: {'ok': True, 'step_retries_total': 2, 'history_preserved': True, 'tool_calls_preserved': True} {'ok': True, 'tool_calls': 1, 'verifier_status': 'train_exact_test_ok'} {'ok': True, 'outcomes': ['guard_rejected', 'executed'], 'executed_calls': 1}
Scripted CPU agent-loop self-test PASS: {'ok': True, 'tool_feedback': "dict_keys(['train', 'test'])\n", 'verifier_status': 'train_exact_test_ok'}
Resume: completed 0 pending 5
batch 1: active=4 | wave=0.0s | caps=12288..12288
batch 1: active=2 | wave=0.0s | caps=11688..11988
batch 1: active=1 | wave=0.0s | caps=11088..11688
batch 1: active=1 | wave=0.0s | caps=11388..11388
batch 1: active=1 | wave=0.0s | caps=11088..11088
batch 1: active=1 | wave=0.0s | caps=10788..10788
batch 1: active=1 | wave=0.0s | caps=10488..10488
batch 1: active=1 | wave=0.0s | caps=10188..10188
batch 1: active=1 | wave=0.0s | caps=9888..9888
batch 1: active=1 | wave=0.0s | caps=9588..9588
batch 1: active=1 | wave=0.0s | caps=928

In [9]:
# =====================================================================================
# CELL 9 — CONTRACT-FILTERED COVERAGE + PIV SCORING + VERIFIED CANDIDATE REGISTRY
# =====================================================================================
raw_records=read_existing_records(); compatible_records,checkpoint_audit=audit_checkpoint_records(raw_records)
with open(WORK_ROOT/"checkpoint_audit.json","w") as f: json.dump(checkpoint_audit,f,indent=2)
if not compatible_records: raise RuntimeError("No compatible PIV records")
# Latest terminal record per active task.
latest_map={}
for r in compatible_records:
    if r.get("status") in {"complete","failed"}: latest_map[r["task_id"]]=r
latest=pd.DataFrame([latest_map[t] for t in ACTIVE_TASK_IDS if t in latest_map])
complete_set={t for t,r in latest_map.items() if r.get("status")=="complete"}
failed_set={t for t,r in latest_map.items() if r.get("status")=="failed"}
missing=sorted(ACTIVE_TASK_SET-set(latest_map))
coverage={"complete_tasks":len(complete_set),"failed_tasks":len(failed_set),"expected_tasks":len(ACTIVE_TASK_IDS),
          "missing_tasks":missing,"run_complete":len(missing)==0 and len(failed_set)==0}
run_complete=coverage["run_complete"]
print("Checkpoint audit:",json.dumps(checkpoint_audit,indent=2)); print("Coverage:",json.dumps(coverage,indent=2))

# Score every public-eval test output; missing/rejected program = no PIV candidate.
rows=[]; registry=[]
weight_map=dict(zip(eval_weights["task_id"],eval_weights["weight"]))
quartile_map=dict(zip(eval_weights["task_id"],eval_weights["test_relevance_quartile"]))
for tid in ACTIVE_TASK_IDS:
    r=latest_map.get(tid,{})
    outs=r.get("test_outputs",[]) if r.get("all_train_exact") else []
    for qi,target in enumerate(eval_solutions[tid]):
        pred=outs[qi] if qi<len(outs) and is_rectangular_grid(outs[qi]) else None
        exact=int(pred is not None and grids_equal(pred,target))
        rows.append({"task_id":tid,"test_index":qi,"piv_valid":int(pred is not None),"piv_exact":exact,
                     "train_exact":int(bool(r.get("all_train_exact"))),"candidate_status":r.get("candidate_status"),
                     "weight":weight_map[tid],"quartile":quartile_map[tid]})
        if pred is not None:
            registry.append({"task_id":tid,"test_index":qi,"grid":pred,"canonical_sha256":hashlib.sha256(json.dumps(pred,separators=(",",":"),sort_keys=False).encode()).hexdigest(),
                "source":"qwen3.8_27b_piv_p0","run_id":r.get("run_id"),"record_id":r.get("record_id"),"seed":r.get("seed"),
                "program_sha256":r.get("final_code_sha256"),"train_exact":True,"piv_source_commit":PIV_SOURCE_COMMIT,
                "generation_contract_sha256":r.get("generation_contract_sha256")})
scores=pd.DataFrame(rows); registry_df=pd.DataFrame(registry)
scores.to_csv(WORK_ROOT/"task_output_scores.csv",index=False); registry_df.to_csv(WORK_ROOT/"candidate_registry.csv",index=False)

def weighted_mean(x,w):
    x=np.asarray(x,dtype=float); w=np.asarray(w,dtype=float); return float(np.sum(x*w)/np.sum(w)) if np.sum(w)>0 else float("nan")
raw_acc=float(scores.piv_exact.mean()); weighted_acc=weighted_mean(scores.piv_exact,scores.weight)
valid_rate=float(scores.piv_valid.mean())
train_perfect_rate=float(np.mean([
    bool(latest_map.get(tid,{}).get("all_train_exact")) for tid in ACTIVE_TASK_IDS
]))
mode_summary=pd.DataFrame([{"branch":"piv_p0","output_weighted_exact":raw_acc,"propensity_weighted_diagnostic":weighted_acc,
                            "valid_grid_rate":valid_rate,"train_perfect_program_rate":train_perfect_rate,
                            "tasks":len(ACTIVE_TASK_IDS),"test_outputs":len(scores)}])
mode_summary.to_csv(WORK_ROOT/"mode_summary.csv",index=False)

task_scores=scores.groupby("task_id")["piv_exact"].min().rename("piv_all_queries").reset_index()
task_scores.to_csv(WORK_ROOT/"strict_task_level_scores.csv",index=False)
print(mode_summary.to_string(index=False))
print("STRICT task-normalized exact:",float(task_scores.piv_all_queries.mean()))
print("Verified programs:",int(latest["all_train_exact"].fillna(False).sum()),"/",len(latest))


Checkpoint audit: {
  "raw_rows": 48,
  "compatible_rows": 48,
  "stale_rows": 0,
  "foreign_rows": 0,
  "stale_preview": [],
  "foreign_preview": []
}
Coverage: {
  "complete_tasks": 48,
  "failed_tasks": 0,
  "expected_tasks": 48,
  "missing_tasks": [],
  "run_complete": true
}
branch  output_weighted_exact  propensity_weighted_diagnostic  valid_grid_rate  train_perfect_program_rate  tasks  test_outputs
piv_p0                    0.0                             0.0              0.0                         0.0     48            56
STRICT task-normalized exact: 0.0
Verified programs: 0 / 48


In [10]:
# =====================================================================================
# CELL 10 — SOURCE-ALIGNED PROXY / EDA BREAKDOWNS
# =====================================================================================
score_task=scores.groupby("task_id").agg(piv_exact=("piv_exact","mean"),valid=("piv_valid","mean"),train_exact=("train_exact","max")).reset_index()
cat=profile_task_table.merge(score_task,on="task_id",how="left")
CATEGORY_MASKS={
 "ALL":lambda d:np.ones(len(d),dtype=bool),"tier1":lambda d:(d.proxy_tier==1).to_numpy(),"tier2":lambda d:(d.proxy_tier==2).to_numpy(),
 "tier3":lambda d:(d.proxy_tier==3).to_numpy(),"easy_static":lambda d:d.proxy_easy_static.astype(bool).to_numpy(),
 "rare":lambda d:d.proxy_rare.astype(bool).to_numpy(),"multi_query":lambda d:(d.proxy_n_queries>1).to_numpy(),
 "dynamic":lambda d:d.proxy_dynamic.astype(bool).to_numpy(),"high_ratio":lambda d:d.proxy_high_ratio.astype(bool).to_numpy(),
 "extreme_ratio":lambda d:d.proxy_extreme_ratio.astype(bool).to_numpy(),
}
rows=[]
for name,fn in CATEGORY_MASKS.items():
    sub=cat.loc[fn(cat)]
    rows.append({"category":name,"tasks":len(sub),"mean_output_exact":float(sub.piv_exact.mean()) if len(sub) else np.nan,
                 "mean_valid":float(sub.valid.mean()) if len(sub) else np.nan,"train_exact_task_rate":float(sub.train_exact.mean()) if len(sub) else np.nan})
category_scores=pd.DataFrame(rows); category_scores.to_csv(WORK_ROOT/"category_scores.csv",index=False); display(category_scores)


,category,tasks,mean_output_exact,mean_valid,train_exact_task_rate
0,ALL,48,0.0,0.0,0.0
1,tier1,32,0.0,0.0,0.0
2,tier2,12,0.0,0.0,0.0
3,tier3,4,0.0,0.0,0.0
4,easy_static,16,0.0,0.0,0.0
5,rare,30,0.0,0.0,0.0
6,multi_query,8,0.0,0.0,0.0
7,dynamic,8,0.0,0.0,0.0
8,high_ratio,5,0.0,0.0,0.0
9,extreme_ratio,4,0.0,0.0,0.0


In [11]:
# =====================================================================================
# CELL 11 — AGENT LOOP / FAILURE TAXONOMY
# =====================================================================================
failure_rows=[]
for _,r in latest.iterrows():
    failure_rows.append({"task_id":r.get("task_id"),"status":r.get("status"),"candidate_status":r.get("candidate_status"),
        "stop_reason":r.get("stop_reason"),"model_turns":r.get("model_turns"),"tool_calls":r.get("tool_calls"),
        "tool_errors":r.get("tool_errors"),"tool_timeouts":r.get("tool_timeouts"),"worker_restarts":r.get("worker_restarts"),
        "generated_tokens":r.get("generated_tokens"),"max_rendered_prompt_tokens":r.get("max_rendered_prompt_tokens"),
        "all_train_exact":r.get("all_train_exact"),"test_execution_ok":r.get("test_execution_ok"),"any_test_output_ok":r.get("any_test_output_ok"),"source_retries":r.get("source_retries"),"step_retries_total":r.get("step_retries_total"),"budget_denials":r.get("budget_denials"),"verifier_status":r.get("verifier_status")})
failure_df=pd.DataFrame(failure_rows); failure_df.to_csv(WORK_ROOT/"agent_diagnostics.csv",index=False)
if len(failure_df):
    print("Candidate status:\n",failure_df.candidate_status.value_counts(dropna=False).to_string())
    print("Stop reason:\n",failure_df.stop_reason.value_counts(dropna=False).to_string())
    print("Mean tool calls:",failure_df.tool_calls.mean(),"| mean generated tokens:",failure_df.generated_tokens.mean())

# SMOKE_GPU is budget-matched to PROXY48. It is a viability/feasibility gate, not a score run.
CORE_BUDGET_DEATH_REASONS={
    "invalid_finish_reason",
    "agent_budget_exhausted",
    "context_or_token_budget_exhausted",
}
# LOCAL stricter accounting: failing to finalize after the six-call P0 tool budget is also a budget death.
ALL_BUDGET_DEATH_REASONS=CORE_BUDGET_DEATH_REASONS | {"tool_budget_exhausted_after_nudge"}

def _json_list(value):
    if isinstance(value,list): return value
    if value is None: return []
    if isinstance(value,str):
        try:
            x=json.loads(value)
            return x if isinstance(x,list) else [x]
        except Exception:
            return [value]
    return [value]

def _death_token_summary(values):
    vals=[float(x) for x in values if x is not None and not pd.isna(x)]
    if not vals:
        return {"count":0,"min":None,"median":None,"p90":None,"max":None,"values":[]}
    return {
        "count":len(vals),"min":float(np.min(vals)),"median":float(np.median(vals)),
        "p90":float(np.percentile(vals,90)),"max":float(np.max(vals)),"values":[int(x) for x in vals],
    }

def compute_smoke_gate(records, *, wave_rows=None, runtime_info=None, smoke_task_count=None):
    smoke_task_count=int(smoke_task_count or len(records) or 1)
    recs=list(records)
    stops=[str(r.get("stop_reason","")) for r in recs]
    budget_dead=[s in ALL_BUDGET_DEATH_REASONS for s in stops]
    core_budget_dead=[s in CORE_BUDGET_DEATH_REASONS for s in stops]
    final_program=[s=="final_program" for s in stops]
    death_tokens=[r.get("generated_tokens") for r,d in zip(recs,budget_dead) if d]

    _terminal_ids=[str(r.get("task_id")) for r in recs if r.get("task_id") is not None]
    terminal_coverage_complete=bool(
        len(recs)==smoke_task_count
        and len(set(_terminal_ids))==smoke_task_count
        and all(str(r.get("status","")) in {"complete","failed"} for r in recs)
    )
    first_finish=[]; infra_bad=[]; tool_engaged=[]; train_exact=[]
    for r in recs:
        fr=_json_list(r.get("finish_reasons",[]))
        first_finish.append(str(fr[0]).lower() if fr else "missing")
        tool_engaged.append(int(float(r.get("tool_calls",0) or 0)>0))
        train_exact.append(bool(r.get("all_train_exact")))
        infra_bad.append(
            str(r.get("stop_reason","")) in {"no_output","global_deadline","global_deadline_projected"}
            or str(r.get("status",""))=="failed"
        )

    first_turn_length_rate=float(np.mean([x=="length" for x in first_finish])) if first_finish else 1.0
    budget_death_rate=float(np.mean(budget_dead)) if recs else 1.0
    core_budget_death_rate=float(np.mean(core_budget_dead)) if recs else 1.0
    final_program_rate=float(np.mean(final_program)) if recs else 0.0
    tool_engagement_rate=float(np.mean(tool_engaged)) if recs else 0.0
    train_exact_rate=float(np.mean(train_exact)) if recs else 0.0
    infra_failure_rate=float(np.mean(infra_bad)) if recs else 1.0

    waves=list(wave_rows or [])
    wave_secs=[float(w.get("wave_sec",0.0) or 0.0) for w in waves]
    wave_sec_total=float(sum(wave_secs))
    spts=[float(w["wave_sec"])/float(w["max_output_tokens"]) for w in waves
          if w.get("max_output_tokens") and float(w.get("wave_sec",0) or 0)>0]
    measured_spt=float(np.median(spts)) if spts else None
    wave_summary={
        "count":len(wave_secs),
        "measured_sec_per_output_token_median":measured_spt,
        "measured_sec_per_output_token_max":(float(np.max(spts)) if spts else None),
        "projected_max_wave_minutes":((TOTAL_GENERATED_TOKEN_CAP*measured_spt)/60.0 if measured_spt else None),
        "sum_sec":wave_sec_total,
        "mean_sec":float(np.mean(wave_secs)) if wave_secs else None,
        "median_sec":float(np.median(wave_secs)) if wave_secs else None,
        "p90_sec":float(np.percentile(wave_secs,90)) if wave_secs else None,
        "max_sec":float(np.max(wave_secs)) if wave_secs else None,
    }

    scale=float(len(PROXY48_TASK_IDS))/float(smoke_task_count)
    projected_generation_hours=(wave_sec_total*scale)/3600.0 if waves else None
    projected_total_hours=None
    projected_total_hours_20pct_margin=None
    if runtime_info:
        pre_loop_sec=float(runtime_info.get("pre_loop_elapsed_sec",0.0) or 0.0)
        smoke_loop_sec=float(runtime_info.get("loop_elapsed_sec",0.0) or 0.0)
        projected_total_hours=(pre_loop_sec + smoke_loop_sec*scale)/3600.0
        projected_total_hours_20pct_margin=(pre_loop_sec + smoke_loop_sec*scale*1.20)/3600.0

    budget_pass=bool(budget_death_rate<=0.50)
    transport_pass=bool(infra_failure_rate==0.0)
    wallclock_pass=bool(projected_total_hours_20pct_margin is not None and projected_total_hours_20pct_margin<=10.0)
    coverage_pass=bool(terminal_coverage_complete)

    gate={
        "first_turn_finish_reasons":first_finish,
        "first_turn_length_rate_diagnostic_only":first_turn_length_rate,
        "core_budget_death_reasons":sorted(CORE_BUDGET_DEATH_REASONS),
        "all_budget_death_reasons":sorted(ALL_BUDGET_DEATH_REASONS),
        "core_budget_death_rate":core_budget_death_rate,
        "budget_death_rate":budget_death_rate,
        "final_program_rate":final_program_rate,
        "tokens_at_budget_death":_death_token_summary(death_tokens),
        "tool_engagement_rate":tool_engagement_rate,
        "all_train_exact_rate":train_exact_rate,
        "infrastructure_failure_rate":infra_failure_rate,
        "wave_runtime":wave_summary,
        "projected_proxy48_generation_hours":projected_generation_hours,
        "projected_proxy48_total_hours":projected_total_hours,
        "projected_proxy48_total_hours_20pct_margin":projected_total_hours_20pct_margin,
        "terminal_records":len(recs),"expected_terminal_records":smoke_task_count,
        "terminal_coverage_complete":terminal_coverage_complete,
        "budget_viability_pass":budget_pass,
        "transport_viability_pass":transport_pass,
        "wallclock_viability_pass":wallclock_pass,
        "coverage_pass":coverage_pass,
    }
    gate["proxy48_advisory_pass"]=bool(budget_pass and transport_pass and wallclock_pass and coverage_pass)
    gate["rule"]=(
        "Do not run PROXY48 if budget_death_rate > 0.50, infrastructure failures are present, "
        "projected_proxy48_total_hours_20pct_margin > 10.0, or any SMOKE task lacks a terminal record."
    )
    return gate

# PREFLIGHT regression for the exact blind spot raised in the external review:
# turn 1 is a short STOP/tool action, turn 2 dies at the generation budget.
_scenario_g=[
    {"task_id":f"g{i}","status":"complete","stop_reason":"invalid_finish_reason",
     "finish_reasons":["stop","length"],"generated_tokens":12288,
     "tool_calls":1,"all_train_exact":False}
    for i in range(8)
]
_scenario_g_gate=compute_smoke_gate(_scenario_g,wave_rows=[],runtime_info=None,smoke_task_count=8)
assert _scenario_g_gate["first_turn_length_rate_diagnostic_only"]==0.0
assert _scenario_g_gate["budget_death_rate"]==1.0
assert not _scenario_g_gate["budget_viability_pass"]
# v1.6: a missing terminal record must block promotion even when every present record looks healthy.
_scenario_cov=[{"task_id":f"c{i}","status":"complete","stop_reason":"final_program","finish_reasons":["stop","stop"],
                "generated_tokens":3000,"tool_calls":2,"all_train_exact":True} for i in range(7)]
_scenario_cov_gate=compute_smoke_gate(_scenario_cov,wave_rows=[{"wave_sec":100.0,"max_output_tokens":500}],
                                      runtime_info={"pre_loop_elapsed_sec":300,"loop_elapsed_sec":600},smoke_task_count=8)
assert _scenario_cov_gate["budget_viability_pass"] and _scenario_cov_gate["wallclock_viability_pass"]
assert not _scenario_cov_gate["coverage_pass"] and not _scenario_cov_gate["proxy48_advisory_pass"]
assert abs(_scenario_cov_gate["wave_runtime"]["measured_sec_per_output_token_median"]-0.2)<1e-9
_scenario_dup=[dict(x) for x in _scenario_cov] + [dict(_scenario_cov[0])]
_scenario_dup_gate=compute_smoke_gate(
    _scenario_dup,wave_rows=[{"wave_sec":100.0,"max_output_tokens":500}],
    runtime_info={"pre_loop_elapsed_sec":300,"loop_elapsed_sec":600},smoke_task_count=8
)
assert not _scenario_dup_gate["terminal_coverage_complete"] and not _scenario_dup_gate["proxy48_advisory_pass"]
with open(WORK_ROOT/"smoke_gate_policy_selftest.json","w") as _f:
    json.dump({"scenario_G":_scenario_g_gate,"scenario_missing_terminal_record":_scenario_cov_gate,
               "scenario_duplicate_terminal_id":_scenario_dup_gate},_f,indent=2)
print("SMOKE gate Scenario-G PREFLIGHT self-test PASS")

if RUN_PROFILE=="SMOKE_GPU":
    _latest_rows={str(r["task_id"]):r.to_dict() for _,r in latest.iterrows()}
    _records=[_latest_rows[str(t)] for t in ACTIVE_TASK_IDS if str(t) in _latest_rows]

    _wave_rows=[]
    if WAVE_METRICS_PATH.exists():
        for _ln in WAVE_METRICS_PATH.read_text(encoding="utf-8").splitlines():
            if not _ln.strip(): continue
            try:
                _w=json.loads(_ln)
                if _w.get("run_id")==RUN_ID: _wave_rows.append(_w)
            except Exception: pass

    _runtime_info=None
    if AGENT_LOOP_RUNTIME_PATH.exists():
        try:
            _runtime_info=json.loads(AGENT_LOOP_RUNTIME_PATH.read_text(encoding="utf-8"))
        except Exception:
            _runtime_info=None

    SMOKE_GATE=compute_smoke_gate(
        _records,wave_rows=_wave_rows,runtime_info=_runtime_info,smoke_task_count=len(ACTIVE_TASK_IDS)
    )
    # v1.6: record what SMOKE8 actually contains so the projection can be read against Proxy-48's mix.
    _pf=proxy_source_feat.set_index("task_id")
    SMOKE_GATE["smoke_task_composition"]={
        "task_ids":list(ACTIVE_TASK_IDS),
        "tier_counts":{f"tier{k}":int(sum(int(_pf.loc[t,"proxy_tier"])==k for t in ACTIVE_TASK_IDS)) for k in (1,2,3)},
        "multi_query":int(sum(int(_pf.loc[t,"proxy_n_queries"])>1 for t in ACTIVE_TASK_IDS)),
        "proxy48_tier_counts":{f"tier{k}":int(sum(int(_pf.loc[t,"proxy_tier"])==k for t in PROXY48_TASK_IDS)) for k in (1,2,3)},
    }
    with open(WORK_ROOT/"smoke_gate.json","w") as _f:
        json.dump(SMOKE_GATE,_f,indent=2)
    print("SMOKE viability/feasibility gate:",json.dumps(SMOKE_GATE,indent=2))
    if not SMOKE_GATE["proxy48_advisory_pass"]:
        print("SMOKE ADVISORY: DO NOT RUN PROXY48 YET. Revisit P0 budget/effort/context/scheduling first.")


Candidate status:
 candidate_status
mock_rejected    48
Stop reason:
 stop_reason
mock_preflight    48
Mean tool calls: 0.0 | mean generated tokens: 0.0
SMOKE gate Scenario-G PREFLIGHT self-test PASS


In [12]:
# =====================================================================================
# CELL 12 — WRITE SOURCE-STYLE PIV REPLAY PAYLOAD (None when train gate rejects)
# =====================================================================================
pred_piv={}
for tid in ACTIVE_TASK_IDS:
    r=latest_map.get(tid,{})
    outs=r.get("test_outputs",[]) if r.get("all_train_exact") else []
    pred_piv[tid]=[]
    for qi in range(len(eval_challenges[tid]["test"])):
        candidate=outs[qi] if qi<len(outs) and is_rectangular_grid(outs[qi]) else None
        # PiVOTools source leaves attempt_1 empty when final code fails the train gate.
        # We keep None here so no schema filler can ever masquerade as a model candidate in offline analysis.
        pred_piv[tid].append({"attempt_1":candidate,"attempt_2":None})
with open(WORK_ROOT/"predictions_piv_replay.json","w") as f: json.dump(pred_piv,f)
print("Wrote predictions_piv_replay.json")


Wrote predictions_piv_replay.json


In [13]:
# =====================================================================================
# CELL 13 — REPLAY SELF-CHECK: saved source-style payload agrees with candidate exact score
# =====================================================================================
pass1=[]
for tid in ACTIVE_TASK_IDS:
    for qi,target in enumerate(eval_solutions[tid]):
        a=pred_piv[tid][qi]["attempt_1"]
        pass1.append(int(a is not None and grids_equal(a,target)))
replay=float(np.mean(pass1))
assert abs(replay-raw_acc)<1e-12,(replay,raw_acc)
print("Replay self-check PASS:",replay)


Replay self-check PASS: 0.0


In [14]:
# =====================================================================================
# CELL 14 — PIV METADATA + SOURCE-LEVEL ALIGNMENT REPORT
# =====================================================================================
total_elapsed_sec=time.perf_counter()-notebook_start
source_verification={
 "source_commit":PIV_SOURCE_COMMIT,
 "external_sources":PIV_EXTERNAL_SOURCES,
 "verified_source_files":PIV_SOURCE_FILES,
 "source_faithful":{
   "persistent_repl":True,"puzzle_preloaded":True,"tool_errors_to_model":True,"source_style_retries":True,"self_contained_solution":True,
   "fresh_train_verifier":True,"all_train_exact_gate":True,"separate_test_execution_state":True,"test_outputs_from_execution":True,"single_lineage_p0":True,
 },
 "intentional_local_adaptations":{
   "openai_responses_server":"replaced by direct vLLM + Qwen official tool chat template",
   "docker_ipybox":"replaced by persistent local subprocess + hard kill/restart + Python audit-hook host isolation",
   "source_timeout_120s":"P0 starts at 10s tool / 20s verifier per portability note",
   "source_public_model":"Qwen3.8-27B-FP8, xhigh, preserve_thinking",
   "grid_gate":"strengthened to rectangular integer 0-9 and 1..30 dimensions",
   "budget":"6 executed tool calls (guard-rejected snippets not charged), 12 total model turns, 12,288 total generated tokens/task; max 2 inner step retries + max 2 outer source-style retries, all sharing the same total budgets",
   "finish_reason_gate":"length/incomplete generations retry the same tool-loop step only while total/context budget remains; a length event that consumes the remaining total budget cannot be rescued by retry",
   "per_turn_ceiling":"up to the remaining 12,288-token P0 total, also bounded by context headroom; source config has no explicit per-step output cap",
   "partial_test_outputs":"train-perfect programs retain successful test outputs even if another test input raises",
   "batch_caps":"per-request max_tokens; no minimum-cap cross-task starvation",
   "approved_library_probe":AVAILABLE_P0_LIBRARIES,
   "v1_6_local_fixes":[
     "safety reserve 60 min + per-wave projected-time pre-check (global_deadline_projected, resumable)",
     "stratified SMOKE8 (1 tier3 / 2 tier2 / 5 tier1) instead of first-match selection that took all four tier3 tasks",
     "structured malformed-tool step-retry counter resets only after a valid structured tool turn",
     "guard-rejected tool snippets are not charged to the 6-call budget; fenced parameter payloads unwrapped",
     "tool output sanitized against chat-control strings before re-entering the prompt",
     "length-capped final turn with a closed, parseable solution() block is accepted (salvaged_final_on_length)",
     "malformed_tool requires an actual <tool_call>",
     "SMOKE gate requires complete terminal coverage; reports measured s/token and projected max wave minutes; wall-clock gate uses the 20% margin",
     "PREFLIGHT drives the real scheduler with a scripted fake LLM",
   ],
   "prefix_caching":("enabled" if ENABLE_PREFIX_CACHING else "disabled (Q1-frozen engine fingerprint)"),
 },
 "not_in_p0":["reviewer","second lineage","PTQ deterministic repair","NVARC seeding","judge","evolution","state compaction ablation"],
}
with open(WORK_ROOT/"source_level_verification.json","w") as f: json.dump(source_verification,f,indent=2)
_smoke_gate_metadata=None
if (WORK_ROOT/"smoke_gate.json").exists():
    try: _smoke_gate_metadata=json.loads((WORK_ROOT/"smoke_gate.json").read_text(encoding="utf-8"))
    except Exception: _smoke_gate_metadata=None

run_metadata={
 "experiment":EXPERIMENT_ID,"run_profile":RUN_PROFILE,"task_set":TASK_SET,"accuracy_is_real":ACCURACY_IS_REAL,"run_id":RUN_ID,
 "active_task_ids":ACTIVE_TASK_IDS,"proxy48_ordered_sha256":PROXY48_ORDERED_SHA256,"piv_source_commit":PIV_SOURCE_COMMIT,
 "generation_contract_version":GENERATION_CONTRACT_VERSION,"parser_contract_version":PARSER_CONTRACT_VERSION,
 "prompt_contract_version":PROMPT_CONTRACT_VERSION,"seed":SEED,"model_path":str(MODEL_PATH),"model_type":model_cfg.get("model_type"),
 "runtime_stack":RUNTIME_STACK,"model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
 "engine_config_sha256":ENGINE_CONFIG_SHA256,"chat_template_sha256":template_sha,"tool_schema_sha256":TOOL_SCHEMA_SHA256,
 "agent_budget":AGENT_BUDGET,"agent_budget_sha256":AGENT_BUDGET_SHA256,"sampling":PIV_SAMPLING,"sampling_config_sha256":SAMPLING_CONFIG_SHA256,
 "model_load_sec":model_load_sec,"warmup_sec":warmup_sec,"total_elapsed_sec":total_elapsed_sec,"max_model_len":MAX_MODEL_LEN,
 "coverage":coverage,"checkpoint_audit":checkpoint_audit,"run_complete":run_complete,
 "output_weighted_exact":raw_acc,"propensity_weighted_diagnostic":weighted_acc,"strict_task_exact":float(task_scores.piv_all_queries.mean()),
 "valid_grid_rate":valid_rate,"train_perfect_program_rate":train_perfect_rate,"candidate_registry_rows":len(registry_df),
 "source_verification":source_verification,"smoke_gate":_smoke_gate_metadata,
}
with open(WORK_ROOT/"run_metadata.json","w") as f: json.dump(run_metadata,f,indent=2,default=str)

report=[]
report += [f"# {EXPERIMENT_ID} — {RUN_PROFILE}","",f"- Run complete: **{run_complete}**",f"- Accuracy is real: **{ACCURACY_IS_REAL}**",
           f"- Frozen tasks: **{len(ACTIVE_TASK_IDS)}**",f"- PIV source commit: `{PIV_SOURCE_COMMIT}`",
           f"- Output-weighted exact: **{raw_acc:.4f}**",f"- Strict task exact: **{float(task_scores.piv_all_queries.mean()):.4f}**",
           f"- Train-perfect program rate: **{train_perfect_rate:.4f}**",f"- Valid-grid rate: **{valid_rate:.4f}**","",
           "## Causal intervention","Persistent Python REPL + model/tool history + self-contained solution(grid) + fresh all-train exact gate.","",
           "## P0 exclusions","No reviewer, no PTQ repair, no second lineage, no NVARC seeding, no judge, no evolution.","",
           "## Source-level verification","See `source_level_verification.json` and `SOURCE_ALIGNMENT.json` for SOURCE vs LOCAL boundaries."]
(WORK_ROOT/"REPORT_FOR_CHATGPT.md").write_text("\n".join(report),encoding="utf-8")
print("\n".join(report))


# PIV-Q27-P0-v1.7 — PREFLIGHT

- Run complete: **True**
- Accuracy is real: **False**
- Frozen tasks: **48**
- PIV source commit: `b3d4494892c6d85fa2a35d548271fb02ea3e191b`
- Output-weighted exact: **0.0000**
- Strict task exact: **0.0000**
- Train-perfect program rate: **0.0000**
- Valid-grid rate: **0.0000**

## Causal intervention
Persistent Python REPL + model/tool history + self-contained solution(grid) + fresh all-train exact gate.

## P0 exclusions
No reviewer, no PTQ repair, no second lineage, no NVARC seeding, no judge, no evolution.

## Source-level verification
See `source_level_verification.json` and `SOURCE_ALIGNMENT.json` for SOURCE vs LOCAL boundaries.


In [15]:
# =====================================================================================
# CELL 15 — MANIFEST + BUNDLE
# =====================================================================================
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()
compact_cols=[c for c in ["run_id","record_id","task_id","status","candidate_status","stop_reason","seed","n_train","n_test",
    "model_turns","tool_calls","tool_errors","tool_timeouts","worker_restarts","source_retries","step_retries_total","budget_denials","generated_tokens","max_rendered_prompt_tokens",
    "all_train_exact","test_execution_ok","any_test_output_ok","verifier_status","final_code_sha256","generation_contract_sha256","sampling_config_sha256","agent_budget_sha256",
    "model_artifact_fingerprint_sha256","runtime_stack_sha256","engine_config_sha256","piv_source_commit"] if c in latest.columns]
latest[compact_cols].to_csv(WORK_ROOT/"generation_summary.csv",index=False)
manifest=[]
for p in sorted(WORK_ROOT.rglob('*')):
    if p.is_file() and p.name!="MANIFEST.csv": manifest.append({"file":str(p.relative_to(WORK_ROOT)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
pd.DataFrame(manifest).to_csv(WORK_ROOT/"MANIFEST.csv",index=False)
ZIP_PATH=Path("/kaggle/working")/BUNDLE_NAME
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORK_ROOT.rglob('*')):
        if p.is_file(): zf.write(p,arcname=str(p.relative_to(WORK_ROOT)))
print("="*116); print(EXPERIMENT_ID,RUN_PROFILE,"PIPELINE COMPLETE"); print("Bundle:",ZIP_PATH); print("ZIP MB:",round(ZIP_PATH.stat().st_size/1024**2,2))
print("UPLOAD bundle + executed notebook back to ChatGPT.")


PIV-Q27-P0-v1.7 PREFLIGHT PIPELINE COMPLETE
Bundle: /kaggle/working/PIV_Q27_P0_PREFLIGHT_BUNDLE.zip
ZIP MB: 0.4
UPLOAD bundle + executed notebook back to ChatGPT.


## Safe run sequence — PIV-Q27-P0 v1.7

This notebook defaults to:

```python
RUN_PROFILE = "PREFLIGHT"
```

### 1) PREFLIGHT
CPU-safe. It validates parser/tool XML behavior, persistent local REPL semantics, protected puzzle state,
fresh exact verification, retry policy, Scenario-G and missing-terminal-record SMOKE-gate logic, and — new in v1.6 —
drives the **real** `run_real_piv()` scheduler with a scripted fake LLM (happy path, budget death, malformed-tool
retry accounting, guard-rejection accounting, length salvage, projected-deadline stop). It also checks the pinned
tokenizer/runtime-package stack. It does **not** load Qwen3.8-27B and its score is not a model result.

Expected bundle: `PIV_Q27_P0_PREFLIGHT_BUNDLE.zip`.

### 2) SMOKE_GPU
After PREFLIGHT passes, change only:

```python
RUN_PROFILE = "SMOKE_GPU"
```

Use **4×L4, Internet Off**.

SMOKE uses 8 frozen tasks, **stratified to mirror Proxy-48** (1 tier-3, 2 tier-2, 5 tier-1 incl. easy_static,
hard_tier1, multi_query and dynamic), with the **same P0 ceilings as PROXY48**:
- one Qwen XHIGH lineage/task;
- up to 6 executed Python calls;
- up to 12 model turns;
- 12,288 total generated tokens/task;
- each individual turn may use the remaining total budget, subject to context headroom;
- max 2 source-style inner step retries and max 2 outer retries, all sharing the same totals;
- agent batch size 8.

Expected bundle: `PIV_Q27_P0_SMOKE_GPU_BUNDLE.zip`.

Inspect `smoke_gate.json`. **Do not run PROXY48** unless `proxy48_advisory_pass=true`.

The gate blocks promotion when any of these is true:
- `budget_death_rate > 0.50`;
- infrastructure failures are present (including `global_deadline_projected`);
- projected Proxy-48 total wall-clock **with the 20% margin** is above **10 hours**;
- any SMOKE task lacks a terminal record.

Also read `wave_runtime.measured_sec_per_output_token_median` and `projected_max_wave_minutes`: they are
configuration-specific SMOKE measurements useful for projection, not universal hardware constants.

`first_turn_length_rate_diagnostic_only` is retained only as a diagnostic; it is not a promotion gate.

### 3) PROXY48
Only after the SMOKE bundle and executed notebook are audited, change only:

```python
RUN_PROFILE = "PROXY48"
```

Production keeps the same one-lineage P0 budget used by SMOKE. The global 12-hour Kaggle envelope now has a
**60-minute** safety reserve, and every wave is pre-checked against a projection from measured wave timings so a
late wave can never cross the Kaggle limit and lose `/kaggle/working`. Tasks stopped that way are `failed` /
`global_deadline_projected` and are re-run on resume (`RESET_PROXY48_OUTPUTS=False`).

Expected bundle: `PIV_Q27_P0_PROXY48_BUNDLE.zip`.

Do not add a reviewer, second lineage, PTQ repair, NVARC hints, judge, evolution, or reasoning-effort change
inside this notebook; those are separate controlled experiments.
